# CS116 Sales Forecasting — 4-cluster LightGBM/XGBoost + Smart zero-fill + Hope-style evaluation/export

Flow: train Jan–Nov 2025, valid Dec 2025, predict Jan 2026. Cluster: `cold_start`, `intermittent`, `low_volume`, `regular`.


In [1]:
# ============================================================
# 1. IMPORTS + CONFIG - SIMPLE/STABLE VERSION
# ============================================================
from pathlib import Path
import gc
import json
import warnings
import time

import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception as e:
    HAS_LIGHTGBM = False
    print("LightGBM import failed:", e)

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except Exception as e:
    HAS_XGBOOST = False
    print("XGBoost import failed:", e)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)
pl.Config.set_tbl_cols(120)
pl.Config.set_tbl_rows(30)

# ====== PATH CONFIG ======
# Auto-detect Kaggle/local parquet paths để tránh FileNotFoundError.
def find_parquet_path(keyword: str, default_name: str):
    candidates = []
    for root in [Path("/kaggle/input"), Path(".")]:
        if root.exists():
            candidates.extend(list(root.rglob("*.parquet")))
    matches = [p for p in candidates if keyword.lower() in p.name.lower()]
    if matches:
        return matches[0]
    return Path(default_name)

TRANSACTION_PATH = find_parquet_path("transaction", "transaction_full_2025 (1).parquet")
ITEM_PATH = find_parquet_path("item", "items (2).parquet")
DATA_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")

# Cache mới cho bản đơn giản để không load nhầm cache cũ gây crash.
CACHE_DIR = DATA_DIR / "cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512"
CACHE_DIR.mkdir(exist_ok=True, parents=True)

# ====== TIME CONFIG ======
TRAIN_START = pd.Timestamp("2025-01-01")
VALID_MONTH = pd.Timestamp("2025-12-01")
TEST_MONTH = None
FORECAST_MONTH = pd.Timestamp("2026-01-01")

# ====== MEMORY / CRASH CONFIG ======
# Bản đơn giản: không smart-zero full panel, không full cross join khi tạo features.
MAX_TRAIN_ROWS = 800_000       # set None nếu máy khỏe; giảm còn 300_000 nếu vẫn crash
MAX_FORECAST_PAIRS = None
USE_CACHE = True
RANDOM_STATE = 42

# Chỉ predict bằng model cho các cặp đã từng bán. Full location x all item được fill quantity=1 ở cell export.
FORECAST_CANDIDATE_MODE = "observed_pairs"
USE_SMART_ZERO_FILL = True
ZERO_SAMPLE_WEIGHT = 0.25
ZERO_FILL_CLUSTERS = ["intermittent", "low_volume"]  # Bật zero-fill có kiểm soát; thêm "cold_start" nếu máy đủ RAM

PURCHASE_EVENTS = {
    "purchased", "purchase", "buy", "order", "transaction", "checkout", "complete_purchase"
}

# ====== SIMPLE 4-CLUSTER CONFIG ======
# 4 nhóm nhẹ để giảm RAM/thời gian nhưng vẫn tách được nhóm bán ngắt quãng:
# cold_start: lịch sử quá ít
# intermittent: bán thưa/lâu không bán/density thấp
# low_volume: bán nhỏ nhưng không quá thưa
# regular: còn lại
CLUSTER_VERSION = "SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB"
COLD_ACTIVE_MONTHS_MAX = 2
LOW_MAX_QTY_MAX = 3
LOW_MEAN_QTY_MAX = 2.5
INTERMITTENT_ACTIVE_MONTHS_MAX = 5
INTERMITTENT_HISTORY_MONTHS_MIN = 6
INTERMITTENT_LAST_SALE_GAP_MIN = 3
INTERMITTENT_DENSITY_MAX = 0.55

# Train thêm XGBoost để candidate search so sánh với LightGBM.
# Nếu Kaggle bị chậm/crash, set USE_XGBOOST = False.
USE_XGBOOST = True

TARGET = "sales_qty"

print("4-cluster thresholds:")
print("  COLD_ACTIVE_MONTHS_MAX =", COLD_ACTIVE_MONTHS_MAX)
print("  LOW_MAX_QTY_MAX =", LOW_MAX_QTY_MAX)
print("  LOW_MEAN_QTY_MAX =", LOW_MEAN_QTY_MAX)
print("  INTERMITTENT_ACTIVE_MONTHS_MAX =", INTERMITTENT_ACTIVE_MONTHS_MAX)
print("  INTERMITTENT_HISTORY_MONTHS_MIN =", INTERMITTENT_HISTORY_MONTHS_MIN)
print("  INTERMITTENT_LAST_SALE_GAP_MIN =", INTERMITTENT_LAST_SALE_GAP_MIN)
print("  INTERMITTENT_DENSITY_MAX =", INTERMITTENT_DENSITY_MAX)
print("TRANSACTION_PATH:", TRANSACTION_PATH)
print("ITEM_PATH:", ITEM_PATH)
print("CACHE_DIR:", CACHE_DIR)


4-cluster thresholds:
  COLD_ACTIVE_MONTHS_MAX = 2
  LOW_MAX_QTY_MAX = 3
  LOW_MEAN_QTY_MAX = 2.5
  INTERMITTENT_ACTIVE_MONTHS_MAX = 5
  INTERMITTENT_HISTORY_MONTHS_MIN = 6
  INTERMITTENT_LAST_SALE_GAP_MIN = 3
  INTERMITTENT_DENSITY_MAX = 0.55
TRANSACTION_PATH: /kaggle/input/datasets/dinhmanhhung168/sale-forecasting/transaction_full_2025 (1).parquet
ITEM_PATH: /kaggle/input/datasets/dinhmanhhung168/sale-forecasting/items (2).parquet
CACHE_DIR: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512


## 2. Helper functions

In [2]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================
def timer(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")


def month_lit(ts: pd.Timestamp) -> pl.Expr:
    return pl.lit(pd.Timestamp(ts).to_pydatetime()).cast(pl.Datetime)


def next_month(ts: pd.Timestamp) -> pd.Timestamp:
    return pd.Timestamp(ts) + pd.offsets.MonthBegin(1)


def month_idx_expr(col: str) -> pl.Expr:
    return (pl.col(col).dt.year() * 12 + pl.col(col).dt.month()).cast(pl.Int32)


def month_diff_expr(a_col: str, b_col: str) -> pl.Expr:
    return (month_idx_expr(a_col) - month_idx_expr(b_col)).cast(pl.Int32)


def parse_datetime_expr(col: str) -> pl.Expr:
    # Với parquet thường Datetime/Date cast được trực tiếp; string cũng parse được khi strict=False.
    return pl.col(col).cast(pl.Datetime, strict=False)


def normalize_event_expr(col: str) -> pl.Expr:
    return (
        pl.col(col)
        .cast(pl.Utf8, strict=False)
        .str.to_lowercase()
        .str.strip_chars()
        .str.replace_all(" ", "_")
    )


def maybe_load_pl(name: str):
    path = CACHE_DIR / name
    if USE_CACHE and path.exists():
        timer(f"Load Polars cache: {path}")
        return pl.read_parquet(path)
    return None


def maybe_save_pl(df: pl.DataFrame, name: str):
    if USE_CACHE:
        path = CACHE_DIR / name
        df.write_parquet(path, compression="zstd")
        timer(f"Saved Polars cache: {path}")


def maybe_load_cache(name):
    """Giữ backward-compatible cho các cell model nếu cần pickle pandas."""
    path = CACHE_DIR / name
    if USE_CACHE and path.exists():
        timer(f"Load cache: {path}")
        return pd.read_pickle(path)
    return None


def maybe_save_cache(obj, name):
    if USE_CACHE:
        path = CACHE_DIR / name
        obj.to_pickle(path)
        timer(f"Saved cache: {path}")


def mape_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / np.abs(y_true[mask])) * 100


def wmape_score(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(np.abs(y_true))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(y_true - y_pred)) / denom * 100


def evaluate_predictions(df, y_col=TARGET, pred_col="y_pred_int"):
    if df is None or len(df) == 0:
        return {"rows": 0, "MAPE": np.nan, "WMAPE": np.nan, "MAE": np.nan, "ACC_exact": np.nan, "ACC_within_1": np.nan}
    y = df[y_col].values.astype(float)
    p = df[pred_col].values.astype(float)
    return {
        "rows": len(df),
        "MAPE": mape_score(y, p),
        "WMAPE": wmape_score(y, p),
        "MAE": float(np.mean(np.abs(y - p))),
        "ACC_exact": float(np.mean(y == p)),
        "ACC_within_1": float(np.mean(np.abs(y - p) <= 1)),
    }


def custom_round_low(pred, t01=0.8, t12=1.8):
    pred = np.asarray(pred, dtype=float)
    out = np.rint(pred).astype(int)
    out[pred < t01] = 0
    out[(pred >= t01) & (pred < t12)] = 1
    out = np.maximum(out, 0)
    return out


def month_diff(a, b):
    return (a.dt.year - b.dt.year) * 12 + (a.dt.month - b.dt.month)


def reduce_memory(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if pd.api.types.is_integer_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="integer")
        elif pd.api.types.is_float_dtype(df[c]):
            df[c] = pd.to_numeric(df[c], downcast="float")
    return df


def sample_index_by_cluster(df: pd.DataFrame, mask: pd.Series, max_rows=None, seed=42, cluster_col="demand_cluster") -> pd.Index:
    idx = df.index[mask]
    n_total = len(idx)
    if max_rows is None or n_total <= max_rows:
        return pd.Index(idx)

    rng = np.random.default_rng(seed)
    if cluster_col in df.columns:
        cluster_vals = df.loc[idx, cluster_col]
        counts = cluster_vals.value_counts(dropna=False)
        chosen = []
        for cl, cnt in counts.items():
            n = max(1, int(round(max_rows * cnt / n_total)))
            n = min(int(cnt), n)
            part_idx = cluster_vals.index[cluster_vals.eq(cl)].to_numpy()
            if len(part_idx) > n:
                part_idx = rng.choice(part_idx, size=n, replace=False)
            chosen.append(part_idx)
        chosen = np.concatenate(chosen) if chosen else np.array([], dtype=idx.dtype)
        if len(chosen) > max_rows:
            chosen = rng.choice(chosen, size=max_rows, replace=False)
        rng.shuffle(chosen)
        return pd.Index(chosen)

    chosen = rng.choice(idx.to_numpy(), size=max_rows, replace=False)
    return pd.Index(chosen)


## 3. Load dữ liệu

In [3]:
# ============================================================
# 3. LOAD DATA - POLARS LAZY FRIENDLY
# ============================================================
if not TRANSACTION_PATH.exists():
    raise FileNotFoundError(f"Không thấy file transaction: {TRANSACTION_PATH}")
if not ITEM_PATH.exists():
    raise FileNotFoundError(f"Không thấy file items: {ITEM_PATH}")

txn_schema = pl.read_parquet_schema(TRANSACTION_PATH)
items_schema = pl.read_parquet_schema(ITEM_PATH)

print("txn columns:", list(txn_schema.keys()))
print("items columns:", list(items_schema.keys()))

# Không đọc full transaction vào RAM. Chỉ xem head và dùng scan_parquet ở các cell sau.
txn_head = pl.scan_parquet(TRANSACTION_PATH).head(5).collect()
items_pl = pl.read_parquet(ITEM_PATH)

# Giữ cả price gốc và item_price để tránh nhầm với price giao dịch.
if "price" in items_pl.columns and "item_price" not in items_pl.columns:
    items_pl = items_pl.with_columns(pl.col("price").alias("item_price"))

print("txn head shape:", txn_head.shape)
print("items shape:", items_pl.shape)
display(txn_head)
display(items_pl.head())


txn columns: ['customer_id', 'item_id', 'price', 'location', 'discount', 'bill_id', 'quantity', 'event_type', 'updated_date']
items columns: ['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'brand', 'manufacturer', 'description', 'sale_status', 'size']
txn head shape: (5, 9)
items shape: (29823, 12)


customer_id,item_id,price,location,discount,bill_id,quantity,event_type,updated_date
i32,str,"decimal[38,4]",i32,"decimal[38,4]",i32,i32,str,datetime[μs]
6291331,"""6767000000002""",285000.0000,1053,0.0000,137262467,1,"""Purchase""",2025-04-12 12:52:41.910
5505362,"""2265000000027""",195000.0000,468,20000.0000,138193104,1,"""Purchase""",2025-04-24 14:57:00.773
7694873,"""6497000000004""",312550.0000,363,16450.0000,138230425,1,"""Purchase""",2025-04-24 20:35:40.027
5837076,"""6587000000001""",149000.0000,48,16000.0000,138235471,1,"""Purchase""",2025-04-24 21:12:32.070
6781756,"""3880000000002""",79000.0000,634,0.0000,138239415,1,"""Purchase""",2025-04-25 08:00:46.293


item_id,price,category_l1,category_l2,category_l3,category,brand,manufacturer,description,sale_status,size,item_price
str,"decimal[38,4]",str,str,str,str,str,str,str,i32,str,"decimal[38,4]"
"""0008040000046""",115000.0000,"""Đồ chơi & Sách""","""1Y+""","""Học tập và phát triển tư duy""","""Siêu nhân, robot""","""WinWinToys""","""Không xác định""","""Robo Luồn thun Winwintoys có h…",0,"""Không xác định""",115000.0000
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Dr.Brown's""","""Không xác định""","""Không xác định""",0,"""Không xác định""",99000.0000
"""0007010000886""",102000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Pigeon""","""Pigeon""","""Không xác định""","""Với hơn 60 năm qua, trải qua r…",0,"""Không xác định""",102000.0000
"""0502020340030""",228000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Phụ kiện bình sữa""","""Bình, túi ủ sữa""","""KUKU""","""Không xác định""","""Bình ủ sữa của KuKu của Đài Lo…",0,"""Không xác định""",228000.0000
"""0012050000007""",59000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Đồ bầu""","""Áo ngực bầu""","""Không xác định""","""Không xác định""","""Không xác định""",0,"""Không xác định""",59000.0000


## 4. Monthly aggregation từ transaction thật

In [4]:
# ============================================================
# 4. MONTHLY SALES + EVENT AGGREGATION - POLARS
# ============================================================
monthly_observed = maybe_load_pl("monthly_observed_polars.parquet")

if monthly_observed is None:
    timer("Prepare monthly observed with Polars")
    required_cols = ["location", "item_id", "quantity", "updated_date"]
    missing = [c for c in required_cols if c not in txn_schema]
    if missing:
        raise ValueError(f"Thiếu cột bắt buộc trong transaction: {missing}")

    base_cols = ["location", "item_id", "quantity", "updated_date"]
    optional_cols = [c for c in ["event_type", "bill_id", "customer_id", "price", "discount"] if c in txn_schema]
    scan_cols = list(dict.fromkeys(base_cols + optional_cols))

    work_lf = (
        pl.scan_parquet(TRANSACTION_PATH)
        .select(scan_cols)
        .with_columns([
            pl.col("location").cast(pl.Utf8, strict=False).alias("location"),
            pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
            pl.col("quantity").cast(pl.Float64, strict=False).fill_null(0).alias("quantity"),
            parse_datetime_expr("updated_date").alias("updated_date"),
        ])
        .filter(pl.col("updated_date").is_not_null())
        .with_columns(pl.col("updated_date").dt.truncate("1mo").alias("month"))
    )

    if "event_type" in scan_cols:
        work_lf = work_lf.with_columns(normalize_event_expr("event_type").alias("event_type_norm"))
        event_counts = (
            work_lf.group_by("event_type_norm")
            .agg(pl.len().alias("len"))
            .sort("len", descending=True)
            .head(30)
            .collect()
        )
        print("event_type counts:")
        display(event_counts)
        n_purchase_match = int(
            work_lf.filter(pl.col("event_type_norm").is_in(list(PURCHASE_EVENTS)))
            .select(pl.len())
            .collect()
            .item()
        )
        if n_purchase_match == 0:
            print("WARNING: Không match PURCHASE_EVENTS, dùng quantity > 0 làm purchase rows.")
            purchase_filter = pl.col("quantity") > 0
        else:
            purchase_filter = pl.col("event_type_norm").is_in(list(PURCHASE_EVENTS))
    else:
        work_lf = work_lf.with_columns(pl.lit("purchase").alias("event_type_norm"))
        purchase_filter = pl.col("quantity") > 0

    sales_lf = (
        work_lf
        .filter(purchase_filter & (pl.col("quantity") > 0))
        .with_columns([
            pl.col("price").cast(pl.Float64, strict=False).alias("price") if "price" in scan_cols else pl.lit(None, dtype=pl.Float64).alias("price"),
            pl.col("discount").cast(pl.Float64, strict=False).alias("discount") if "discount" in scan_cols else pl.lit(None, dtype=pl.Float64).alias("discount"),
        ])
    )

    agg_exprs = [
        pl.col("quantity").sum().alias("sales_qty"),
        pl.len().alias("purchase_count"),
    ]
    if "bill_id" in scan_cols:
        agg_exprs.append(pl.col("bill_id").n_unique().alias("unique_bills"))
    if "customer_id" in scan_cols:
        agg_exprs.append(pl.col("customer_id").n_unique().alias("unique_customers"))
    if "price" in scan_cols:
        agg_exprs.append(pl.col("price").mean().alias("avg_txn_price"))
    if "discount" in scan_cols:
        agg_exprs.extend([
            pl.col("discount").sum().alias("discount_sum"),
            pl.col("discount").mean().alias("discount_mean"),
        ])

    monthly_observed = (
        sales_lf
        .group_by(["month", "location", "item_id"])
        .agg(agg_exprs)
        .with_columns([
            pl.lit(1, dtype=pl.Int8).alias("is_observed_sale_month"),
            pl.lit(0, dtype=pl.Int8).alias("is_smart_zero"),
            pl.lit(0, dtype=pl.Int8).alias("is_forecast"),
        ])
        .collect()
    )

    # Optional event counts: merge vào các monthly sale rows hiện có, giống notebook gốc.
    for ev, out_col in [("view_item", "view_item_count"), ("add_to_cart", "add_to_cart_count")]:
        tmp = (
            work_lf
            .filter(pl.col("event_type_norm") == ev)
            .group_by(["month", "location", "item_id"])
            .agg(pl.len().alias(out_col))
            .collect()
        )
        if tmp.height:
            monthly_observed = monthly_observed.join(tmp, on=["month", "location", "item_id"], how="left")
        else:
            monthly_observed = monthly_observed.with_columns(pl.lit(0, dtype=pl.Int32).alias(out_col))

    # Merge static item information.
    if "item_id" not in items_pl.columns:
        raise ValueError("items thiếu item_id")
    item_static_cols = [c for c in ["item_id", "cat_1", "cat_2", "cat_3", "cat", "category", "brand", "item_price", "status"] if c in items_pl.columns]
    item_static = (
        items_pl
        .select([pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id")] + [pl.col(c) for c in item_static_cols if c != "item_id"])
        .unique(subset=["item_id"], keep="first")
    )
    monthly_observed = monthly_observed.join(item_static, on="item_id", how="left")

    for c in ["view_item_count", "add_to_cart_count", "unique_bills", "unique_customers", "discount_sum", "discount_mean", "avg_txn_price"]:
        if c in monthly_observed.columns:
            monthly_observed = monthly_observed.with_columns(pl.col(c).cast(pl.Float64, strict=False).fill_null(0).alias(c))

    if "avg_txn_price" in monthly_observed.columns and "item_price" in monthly_observed.columns:
        monthly_observed = monthly_observed.with_columns(
            pl.when(pl.col("avg_txn_price") > 0)
            .then(pl.col("avg_txn_price"))
            .otherwise(pl.col("item_price").cast(pl.Float64, strict=False))
            .fill_null(0)
            .alias("price")
        )
    elif "item_price" in monthly_observed.columns:
        monthly_observed = monthly_observed.with_columns(pl.col("item_price").cast(pl.Float64, strict=False).fill_null(0).alias("price"))

    maybe_save_pl(monthly_observed, "monthly_observed_polars.parquet")

print("monthly_observed shape:", monthly_observed.shape)
display(monthly_observed.head())
display(monthly_observed.select([pl.col(TARGET).min().alias("min_sales_qty"), pl.col(TARGET).max().alias("max_sales_qty"), pl.col(TARGET).mean().alias("mean_sales_qty")]))


[16:25:10] Prepare monthly observed with Polars
event_type counts:


event_type_norm,len
str,u32
"""purchase""",41470317


[16:26:10] Saved Polars cache: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/monthly_observed_polars.parquet
monthly_observed shape: (12167608, 19)


month,location,item_id,sales_qty,purchase_count,unique_bills,unique_customers,avg_txn_price,discount_sum,discount_mean,is_observed_sale_month,is_smart_zero,is_forecast,view_item_count,add_to_cart_count,category,brand,item_price,price
datetime[μs],str,str,f64,u32,f64,f64,f64,f64,f64,i8,i8,i8,f64,f64,str,str,"decimal[38,4]",f64
2025-01-01 00:00:00,"""370""","""3442000000091""",2.0,2,2.0,2.0,89000.0,80000.0,40000.0,1,0,0,0.0,0.0,"""Quần short bé trai""","""Animo""",129000.0000,89000.0
2025-10-01 00:00:00,"""682""","""1236000000002""",2.0,2,2.0,2.0,185000.0,40000.0,20000.0,1,0,0,0.0,0.0,"""Bobby Fresh_Sơ Sinh""","""Bobby""",205000.0000,185000.0
2025-06-01 00:00:00,"""930""","""5537000000011""",9.0,9,9.0,7.0,74875.319489,1122.1246,124.680511,1,0,0,0.0,0.0,"""Ivenet""","""Ivenet""",75000.0000,74875.319489
2025-11-01 00:00:00,"""180""","""0214000000009""",1.0,1,1.0,1.0,52500.0,0.0,0.0,1,0,0,0.0,0.0,"""Túi rác""","""Soji""",75000.0000,52500.0
2025-02-01 00:00:00,"""259""","""5490000000011""",1.0,1,1.0,1.0,99000.0,0.0,0.0,1,0,0,0.0,0.0,"""Gia vị rắc""","""Ivenet""",99000.0000,99000.0


min_sales_qty,max_sales_qty,mean_sales_qty
f64,f64,f64
1.0,3137.0,5.423062


## 5. Simple 4-cluster leak-safe clustering

Chia 4 nhóm: `cold_start`, `intermittent`, `low_volume`, `regular`. Bản này mở rộng `intermittent` bằng density 0.55 để kéo các pair bán thưa ra khỏi `regular`.


In [5]:
# ============================================================
# 5. SIMPLE 4-CLUSTER LEAK-SAFE CLUSTERING - POLARS
# ============================================================
# Bản nhẹ để tránh crash:
# - Chia 4 nhóm: cold_start, intermittent, low_volume, regular.
# - Không dùng spiky/smooth phức tạp.
# - Cluster của target_month chỉ dùng history month < target_month.
#
# Rule:
# 1) cold_start   : quá ít tháng có bán
# 2) intermittent : bán thưa / lâu không bán / density thấp
# 3) low_volume   : bán ít, quantity nhỏ, nhưng không quá intermittent
# 4) regular      : còn lại
#
# Bản update:
# - Mở rộng intermittent: active_months <= 5 hoặc density <= 0.55.
# - Đổi cache name theo threshold để không load nhầm cache cũ.

import gc
import pandas as pd
import polars as pl

# ============================================================
# Config threshold cho intermittent nếu chưa có ở cell config
# ============================================================
if "INTERMITTENT_ACTIVE_MONTHS_MAX" not in globals():
    INTERMITTENT_ACTIVE_MONTHS_MAX = 5

if "INTERMITTENT_HISTORY_MONTHS_MIN" not in globals():
    INTERMITTENT_HISTORY_MONTHS_MIN = 6

if "INTERMITTENT_LAST_SALE_GAP_MIN" not in globals():
    INTERMITTENT_LAST_SALE_GAP_MIN = 3

if "INTERMITTENT_DENSITY_MAX" not in globals():
    INTERMITTENT_DENSITY_MAX = 0.55


def build_cluster_map_pl(
    observed_pl: pl.DataFrame,
    target_month: pd.Timestamp | None = None,
) -> pl.DataFrame:
    base_cols = [
        "location",
        "item_id",
        "first_month",
        "last_month",
        "active_months",
        "total_qty",
        "mean_qty",
        "median_qty",
        "max_qty",
        "history_months",
        "months_since_last_sale_at_cutoff",
        "density",
        "demand_cluster",
    ]

    if observed_pl is None or observed_pl.height == 0:
        return pl.DataFrame({c: [] for c in base_cols})

    target_ts = pd.Timestamp(target_month) if target_month is not None else None

    # Chỉ dùng tháng có bán thật để tính cluster
    df = (
        observed_pl
        .filter(
            pl.col(TARGET).is_not_null()
            & (pl.col(TARGET) > 0)
            & pl.col("month").is_not_null()
        )
        .select(["location", "item_id", "month", TARGET])
    )

    if df.height == 0:
        return pl.DataFrame({c: [] for c in base_cols})

    # Thống kê nhẹ theo từng pair
    stats = (
        df.group_by(["location", "item_id"])
        .agg([
            pl.col("month").min().alias("first_month"),
            pl.col("month").max().alias("last_month"),
            pl.col("month").n_unique().cast(pl.Int16).alias("active_months"),
            pl.col(TARGET).sum().cast(pl.Float32).alias("total_qty"),
            pl.col(TARGET).mean().cast(pl.Float32).alias("mean_qty"),
            pl.col(TARGET).median().cast(pl.Float32).alias("median_qty"),
            pl.col(TARGET).max().cast(pl.Float32).alias("max_qty"),
        ])
    )

    # cutoff = tháng ngay trước target_month
    # Ví dụ target 2026-01 => cutoff 2025-12
    if target_ts is not None:
        cutoff_idx = target_ts.year * 12 + target_ts.month - 1
        stats = stats.with_columns(
            pl.lit(cutoff_idx).cast(pl.Int32).alias("cutoff_idx")
        )
    else:
        stats = stats.with_columns(
            month_idx_expr("last_month").alias("cutoff_idx")
        )

    stats = (
        stats
        .with_columns([
            month_idx_expr("first_month").alias("first_idx"),
            month_idx_expr("last_month").alias("last_idx"),
        ])
        .with_columns([
            (
                pl.col("cutoff_idx") - pl.col("first_idx") + 1
            )
            .clip(1, 99)
            .cast(pl.Int16)
            .alias("history_months"),

            (
                pl.col("cutoff_idx") - pl.col("last_idx")
            )
            .clip(0, 99)
            .cast(pl.Int16)
            .alias("months_since_last_sale_at_cutoff"),
        ])
        .with_columns([
            (
                pl.col("active_months").cast(pl.Float32)
                / pl.col("history_months").cast(pl.Float32)
            )
            .fill_null(0.0)
            .alias("density")
        ])
    )

    # ============================================================
    # 4-cluster rule
    # Thứ tự rất quan trọng:
    # cold_start -> intermittent -> low_volume -> regular
    # ============================================================

    cond_cold = (
        pl.col("active_months") <= COLD_ACTIVE_MONTHS_MAX
    )

    # Intermittent = đã có nhiều hơn cold-start, nhưng bán thưa/ngắt quãng.
    # Rule này cố tình đặt trước low_volume để các pair bán thưa không bị gom vào low_volume/regular.
    cond_intermit = (
        (~cond_cold)
        & (
            (
                (pl.col("active_months") <= INTERMITTENT_ACTIVE_MONTHS_MAX)
                & (pl.col("history_months") >= INTERMITTENT_HISTORY_MONTHS_MIN)
            )
            |
            (
                (pl.col("months_since_last_sale_at_cutoff") >= INTERMITTENT_LAST_SALE_GAP_MIN)
                & (pl.col("history_months") >= INTERMITTENT_HISTORY_MONTHS_MIN)
            )
            |
            (
                pl.col("density") <= INTERMITTENT_DENSITY_MAX
            )
        )
    )

    # Low volume = quantity nhỏ, nhưng không quá intermittent
    cond_low = (
        (~cond_cold)
        & (~cond_intermit)
        & (
            (pl.col("max_qty") <= LOW_MAX_QTY_MAX)
            |
            (
                (pl.col("median_qty") <= 2)
                & (pl.col("mean_qty") <= LOW_MEAN_QTY_MAX)
            )
        )
    )

    out = (
        stats
        .with_columns(
            pl.when(cond_cold)
            .then(pl.lit("cold_start"))
            .when(cond_intermit)
            .then(pl.lit("intermittent"))
            .when(cond_low)
            .then(pl.lit("low_volume"))
            .otherwise(pl.lit("regular"))
            .alias("demand_cluster")
        )
        .select(base_cols)
    )

    return out


def build_cluster_asof_pl(
    monthly_observed_pl: pl.DataFrame,
    months: list[pd.Timestamp],
) -> pl.DataFrame:
    parts = []

    for m in months:
        m = pd.Timestamp(m)

        # Leak-safe: target month m chỉ dùng history month < m
        hist = monthly_observed_pl.filter(
            pl.col("month") < month_lit(m)
        )

        cm = build_cluster_map_pl(
            hist,
            target_month=m,
        )

        if cm.height:
            cm = cm.with_columns(
                month_lit(m).alias("target_month")
            )
            parts.append(cm)

        del hist, cm
        gc.collect()

    if not parts:
        return pl.DataFrame({
            "location": [],
            "item_id": [],
            "target_month": [],
            "first_month": [],
            "last_month": [],
            "active_months": [],
            "total_qty": [],
            "mean_qty": [],
            "median_qty": [],
            "max_qty": [],
            "history_months": [],
            "months_since_last_sale_at_cutoff": [],
            "density": [],
            "demand_cluster": [],
        })

    return pl.concat(parts, how="vertical_relaxed")


# ============================================================
# Build cluster as-of
# ============================================================
ASOF_MONTHS = list(
    pd.date_range(
        TRAIN_START,
        FORECAST_MONTH,
        freq="MS",
    )
)

# Đổi cache name theo threshold để không load lại cache 3-cluster/4-cluster cũ.
cluster_asof_cache_name = (
    f"cluster_asof_NOLEAK_4CLUSTER_"
    f"cold{COLD_ACTIVE_MONTHS_MAX}_"
    f"intA{INTERMITTENT_ACTIVE_MONTHS_MAX}_"
    f"intH{INTERMITTENT_HISTORY_MONTHS_MIN}_"
    f"intGap{INTERMITTENT_LAST_SALE_GAP_MIN}_"
    f"intDen{str(INTERMITTENT_DENSITY_MAX).replace('.', 'p')}_"
    f"lowMax{LOW_MAX_QTY_MAX}_"
    f"{CLUSTER_VERSION}.parquet"
)

cluster_asof = maybe_load_pl(cluster_asof_cache_name)

if cluster_asof is None:
    timer(f"Build SIMPLE 4-cluster leak-safe as-of maps - {CLUSTER_VERSION}")
    cluster_asof = build_cluster_asof_pl(
        monthly_observed,
        ASOF_MONTHS,
    )
    maybe_save_pl(
        cluster_asof,
        cluster_asof_cache_name,
    )

cluster_map_final = cluster_asof.filter(
    pl.col("target_month") == month_lit(FORECAST_MONTH)
)

print("cluster_asof shape:", cluster_asof.shape)
print("cluster_map_final shape:", cluster_map_final.shape)
print("Cluster cache:", cluster_asof_cache_name)

if cluster_map_final.height:
    print("Forecast pair distribution:")
    forecast_cluster_dist = (
        cluster_map_final
        .group_by("demand_cluster")
        .len()
        .sort("len", descending=True)
    )
    display(forecast_cluster_dist)

    print("Forecast cluster stats:")
    stat_cols = [
        "active_months",
        "history_months",
        "total_qty",
        "mean_qty",
        "median_qty",
        "max_qty",
        "months_since_last_sale_at_cutoff",
        "density",
    ]

    display(
        cluster_map_final
        .group_by("demand_cluster")
        .agg([
            pl.col(c).mean().alias(c)
            for c in stat_cols
            if c in cluster_map_final.columns
        ])
        .sort("demand_cluster")
    )

    # Check để biết đủ 4 nhóm chưa. Không ép nếu data thật không có nhóm nào đó.
    expected_clusters = {"cold_start", "intermittent", "low_volume", "regular"}
    actual_clusters = set(cluster_map_final["demand_cluster"].unique().to_list())
    missing_clusters = sorted(expected_clusters - actual_clusters)
    if missing_clusters:
        print("WARNING: Forecast month chưa xuất hiện đủ 4 cluster:", missing_clusters)
        print("Nếu muốn intermittent rộng hơn nữa, tăng INTERMITTENT_DENSITY_MAX lên 0.60 hoặc giảm INTERMITTENT_HISTORY_MONTHS_MIN.")
    else:
        print("OK: Forecast month có đủ 4 cluster.")


[16:26:10] Build SIMPLE 4-cluster leak-safe as-of maps - SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB
[16:26:48] Saved Polars cache: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/cluster_asof_NOLEAK_4CLUSTER_cold2_intA5_intH6_intGap3_intDen0p55_lowMax3_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB.parquet
cluster_asof shape: (25017692, 14)
cluster_map_final shape: (3144558, 14)
Cluster cache: cluster_asof_NOLEAK_4CLUSTER_cold2_intA5_intH6_intGap3_intDen0p55_lowMax3_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB.parquet
Forecast pair distribution:


demand_cluster,len
str,u32
"""cold_start""",1688434
"""intermittent""",599482
"""regular""",532149
"""low_volume""",324493


Forecast cluster stats:


demand_cluster,active_months,history_months,total_qty,mean_qty,median_qty,max_qty,months_since_last_sale_at_cutoff,density
str,f64,f64,f32,f32,f32,f32,f64,f32
"""cold_start""",1.354469,6.224624,2.213856,1.606823,1.606823,1.759896,4.352227,0.370146
"""intermittent""",4.251232,10.05015,10.421175,2.213089,1.971415,3.690485,2.856489,0.4364
"""low_volume""",7.0592,9.258631,12.878601,1.765675,1.502492,3.302558,0.343197,0.775218
"""regular""",9.473809,10.295201,97.381401,9.654705,8.928663,18.918356,0.142954,0.922816


OK: Forecast month có đủ 4 cluster.


## 6. Smart Zero-Filling có kiểm soát

Bật `USE_SMART_ZERO_FILL = True`. Notebook chỉ tạo zero rows cho các tháng không bán của những cluster trong `ZERO_FILL_CLUSTERS`, mặc định `intermittent` và `low_volume`, để giảm RAM và tránh tạo full location × item.


In [6]:
# ============================================================
# 6. SMART ZERO-FILLING PANEL - CONTROLLED + LAG-SAFE
# ============================================================
# Mục tiêu:
# - Bật zero-fill cho các tháng không có sale thật, nhưng KHÔNG tạo full location x item.
# - Chỉ zero-fill các cluster cần học khoảng không bán: mặc định intermittent + low_volume.
# - Zero rows dùng target = 0, is_smart_zero = 1, is_observed_sale_month = 0.
# - Valid Dec vẫn chỉ đánh giá sale thật ở cell split; zero Dec chỉ giúp tạo lag cho Jan/2026.

if "USE_SMART_ZERO_FILL" not in globals():
    USE_SMART_ZERO_FILL = True

if "ZERO_FILL_CLUSTERS" not in globals():
    ZERO_FILL_CLUSTERS = ["intermittent", "low_volume"]

ZERO_FILL_CLUSTERS = [str(c) for c in ZERO_FILL_CLUSTERS]
print("USE_SMART_ZERO_FILL:", USE_SMART_ZERO_FILL)
print("ZERO_FILL_CLUSTERS:", ZERO_FILL_CLUSTERS)
print("ZERO_SAMPLE_WEIGHT:", globals().get("ZERO_SAMPLE_WEIGHT", None))


def make_forecast_pairs_pl(
    monthly_observed_pl: pl.DataFrame,
    items_pl: pl.DataFrame,
    cluster_map_final_pl: pl.DataFrame,
) -> pl.DataFrame:
    forecast_pairs = cluster_map_final_pl.select(
        ["location", "item_id", "first_month", "demand_cluster"]
    )

    if MAX_FORECAST_PAIRS is not None and forecast_pairs.height > MAX_FORECAST_PAIRS:
        forecast_pairs = forecast_pairs.sample(n=MAX_FORECAST_PAIRS, seed=RANDOM_STATE)
        print("Sample forecast_pairs:", forecast_pairs.height)

    return forecast_pairs


def build_smart_zero_rows_pl(
    monthly_observed_pl: pl.DataFrame,
    cluster_asof_pl: pl.DataFrame,
) -> pl.DataFrame:
    """Tạo zero rows leak-safe từ cluster_asof.

    Với mỗi target_month m, cluster_asof đã chỉ dùng history < m.
    Vì vậy zero row ở tháng m không nhìn thấy sale của chính tháng m.
    """

    if (not USE_SMART_ZERO_FILL) or len(ZERO_FILL_CLUSTERS) == 0:
        return pl.DataFrame()

    key_cols = ["location", "item_id", "month"]

    keep_cols = [
        "location", "item_id", "target_month", "first_month", "last_month",
        "active_months", "total_qty", "mean_qty", "median_qty", "max_qty",
        "history_months", "months_since_last_sale_at_cutoff", "density",
        "demand_cluster",
    ]
    keep_cols = [c for c in keep_cols if c in cluster_asof_pl.columns]

    zero_base = (
        cluster_asof_pl
        .filter(
            (pl.col("target_month") >= month_lit(TRAIN_START))
            & (pl.col("target_month") < month_lit(FORECAST_MONTH))
            & pl.col("demand_cluster").is_in(ZERO_FILL_CLUSTERS)
        )
        .select(keep_cols)
        .rename({"target_month": "month"})
    )

    if zero_base.height == 0:
        return pl.DataFrame()

    observed_keys = (
        monthly_observed_pl
        .select(key_cols)
        .unique()
    )

    zero_rows = (
        zero_base
        .join(observed_keys, on=key_cols, how="anti")
        .with_columns([
            pl.lit(0.0, dtype=pl.Float64).alias(TARGET),
            pl.lit(0, dtype=pl.Int8).alias("is_forecast"),
            pl.lit(1, dtype=pl.Int8).alias("is_smart_zero"),
            pl.lit(0, dtype=pl.Int8).alias("is_observed_sale_month"),
        ])
    )

    return zero_rows


def build_smart_panel_pl(
    monthly_observed_pl: pl.DataFrame,
    items_pl: pl.DataFrame,
    cluster_asof_pl: pl.DataFrame,
    cluster_map_final_pl: pl.DataFrame,
) -> pl.DataFrame:
    timer("Build SMART observed + zero-fill + forecast panel - Polars")

    keep_cols = [
        "location", "item_id", "target_month", "first_month", "last_month",
        "active_months", "total_qty", "mean_qty", "median_qty", "max_qty",
        "history_months", "months_since_last_sale_at_cutoff", "density",
        "demand_cluster",
    ]
    asof_small = cluster_asof_pl.select([c for c in keep_cols if c in cluster_asof_pl.columns])

    # Observed sale rows: cluster của tháng m được tính bằng history < m.
    obs = (
        monthly_observed_pl
        .with_columns(pl.col("month").alias("target_month"))
        .join(asof_small, on=["location", "item_id", "target_month"], how="left")
        .with_columns([
            pl.col("demand_cluster").fill_null("cold_start").alias("demand_cluster"),
            pl.col("first_month").fill_null(pl.col("month")).alias("first_month"),
            pl.lit(0, dtype=pl.Int8).alias("is_forecast"),
            pl.lit(0, dtype=pl.Int8).alias("is_smart_zero"),
            pl.lit(1, dtype=pl.Int8).alias("is_observed_sale_month"),
        ])
        .drop("target_month")
    )

    zero_rows = build_smart_zero_rows_pl(monthly_observed_pl, cluster_asof_pl)
    print("smart zero rows:", f"{zero_rows.height:,}")

    forecast_pairs = make_forecast_pairs_pl(
        monthly_observed_pl,
        items_pl,
        cluster_map_final_pl,
    )

    forecast_rows = forecast_pairs.with_columns([
        month_lit(FORECAST_MONTH).alias("month"),
        pl.lit(1, dtype=pl.Int8).alias("is_forecast"),
        pl.lit(0, dtype=pl.Int8).alias("is_smart_zero"),
        pl.lit(0, dtype=pl.Int8).alias("is_observed_sale_month"),
        pl.lit(None, dtype=pl.Float64).alias(TARGET),
    ])

    parts = [obs]
    if zero_rows.height > 0:
        parts.append(zero_rows)
    parts.append(forecast_rows)

    panel = (
        pl.concat(parts, how="diagonal_relaxed")
        .sort(["location", "item_id", "month", "is_forecast", "is_smart_zero"])
        .unique(subset=["location", "item_id", "month"], keep="last")
    )

    # Join static item info nhẹ, chỉ những cột cần fallback/code.
    item_static_cols = [
        c for c in [
            "item_id", "cat_1", "cat_2", "cat_3", "cat", "category",
            "brand", "item_price", "status",
        ]
        if c in items_pl.columns
    ]

    if item_static_cols:
        item_static = (
            items_pl
            .select(
                [pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id")]
                + [pl.col(c) for c in item_static_cols if c != "item_id"]
            )
            .unique(subset=["item_id"], keep="first")
        )
        suffix_cols = [c for c in item_static_cols if c != "item_id" and c in panel.columns]
        panel = (
            panel
            .drop(suffix_cols, strict=False)
            .join(item_static, on="item_id", how="left")
        )

    # Các cột event/price raw chỉ giữ để tạo lag ở cell feature; null -> 0.
    for c in [
        "purchase_count", "unique_bills", "unique_customers",
        "view_item_count", "add_to_cart_count",
        "discount_sum", "discount_mean", "avg_txn_price", "price",
    ]:
        if c in panel.columns:
            panel = panel.with_columns(
                pl.col(c)
                .cast(pl.Float64, strict=False)
                .fill_null(0)
                .alias(c)
            )

    panel = panel.with_columns([
        pl.col("is_forecast").fill_null(0).cast(pl.Int8).alias("is_forecast"),
        pl.col("is_smart_zero").fill_null(0).cast(pl.Int8).alias("is_smart_zero"),
        pl.col("is_observed_sale_month").fill_null(0).cast(pl.Int8).alias("is_observed_sale_month"),
        pl.col(TARGET).cast(pl.Float64, strict=False).alias(TARGET),
    ])

    return panel


zero_tag = "SMART_ZERO_" + "_".join(ZERO_FILL_CLUSTERS) if USE_SMART_ZERO_FILL else "NO_SMART_ZERO"
panel_cache_name = f"panel_{zero_tag}_{CLUSTER_VERSION}.parquet"

panel_smart = maybe_load_pl(panel_cache_name)

if panel_smart is None:
    panel_smart = build_smart_panel_pl(
        monthly_observed,
        items_pl,
        cluster_asof,
        cluster_map_final,
    )
    maybe_save_pl(panel_smart, panel_cache_name)

print("panel_smart shape:", panel_smart.shape)
print("panel cache:", panel_cache_name)

display(panel_smart.select([
    pl.col("is_observed_sale_month").sum().alias("observed_sale_rows"),
    pl.col("is_smart_zero").sum().alias("smart_zero_rows"),
    pl.col("is_forecast").sum().alias("forecast_rows"),
]))

display(
    panel_smart
    .group_by(["demand_cluster", "is_smart_zero"])
    .len()
    .sort(["demand_cluster", "is_smart_zero"])
)

display(panel_smart.head())


USE_SMART_ZERO_FILL: True
ZERO_FILL_CLUSTERS: ['intermittent', 'low_volume']
ZERO_SAMPLE_WEIGHT: 0.25
[16:26:48] Build SMART observed + zero-fill + forecast panel - Polars
smart zero rows: 2,571,903
[16:28:00] Saved Polars cache: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/panel_SMART_ZERO_intermittent_low_volume_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB.parquet
panel_smart shape: (17884069, 30)
panel cache: panel_SMART_ZERO_intermittent_low_volume_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB.parquet


observed_sale_rows,smart_zero_rows,forecast_rows
i64,i64,i64
12167608,2571903,3144558


demand_cluster,is_smart_zero,len
str,i8,u32
"""cold_start""",0,8343738
"""intermittent""",0,1480875
"""intermittent""",1,1657109
"""low_volume""",0,1614485
"""low_volume""",1,914794
"""regular""",0,3873068


month,location,item_id,sales_qty,purchase_count,unique_bills,unique_customers,avg_txn_price,discount_sum,discount_mean,is_observed_sale_month,is_smart_zero,is_forecast,view_item_count,add_to_cart_count,price,first_month,last_month,active_months,total_qty,mean_qty,median_qty,max_qty,history_months,months_since_last_sale_at_cutoff,density,demand_cluster,category,brand,item_price
datetime[μs],str,str,f64,f64,f64,f64,f64,f64,f64,i8,i8,i8,f64,f64,f64,datetime[μs],datetime[μs],i16,f32,f32,f32,f32,i16,i16,f32,str,str,str,"decimal[38,4]"
2025-01-01 00:00:00,"""1014""","""6048000000107""",1.0,1.0,1.0,1.0,169000.0,30000.0,30000.0,1,0,0,0.0,0.0,169000.0,2025-01-01 00:00:00,null,null,null,null,null,null,null,null,null,"""cold_start""","""Bodysuit đông vải mỏng""","""Animo""",199000.0000
2025-07-01 00:00:00,"""954""","""0029080000004""",1.0,1.0,1.0,1.0,36000.0,0.0,0.0,1,0,0,0.0,0.0,36000.0,2025-01-01 00:00:00,2025-06-01 00:00:00,6,23.0,3.833333,3.5,7.0,6,0,1.0,"""regular""","""Sài Gòn Food""","""Sài Gòn Food""",36000.0000
2025-07-01 00:00:00,"""268""","""2123025000002""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1,0,0.0,0.0,0.0,2025-02-01 00:00:00,2025-05-01 00:00:00,3,3.0,1.0,1.0,1.0,5,1,0.6,"""low_volume""","""Gặm nướu silicone Animo""","""Animo""",189000.0000
2025-05-01 00:00:00,"""1115""","""2027000000001""",4.0,3.0,3.0,3.0,20000.0,0.0,0.0,1,0,0,0.0,0.0,20000.0,2025-04-01 00:00:00,2025-04-01 00:00:00,1,14.0,14.0,14.0,14.0,1,0,1.0,"""cold_start""","""Probi""","""Probi""",25000.0000
2025-09-01 00:00:00,"""728""","""0006040000302""",3.0,3.0,3.0,3.0,290000.0,90000.0,30000.0,1,0,0,0.0,0.0,290000.0,2025-01-01 00:00:00,2025-05-01 00:00:00,2,2.0,1.0,1.0,1.0,8,3,0.25,"""cold_start""","""Bình sữa Pigeon""","""Pigeon""",320000.0000


## 7. Feature engineering nhanh, lag-safe

In [7]:
# ============================================================
# 7. SIMPLE FEATURE ENGINEERING - FAST + LOW RAM + LAG-SAFE
# FIXED: tránh lỗi inf/-inf với cột integer
# ============================================================
import gc
import polars as pl
import polars.selectors as cs

# Chỉ giữ feature ít nhưng đủ mạnh:
# calendar + qty lags + sold lags + gap + vài event lag1 + id/category codes.
# Bỏ rolling/candidate phức tạp nhiều cột để giảm RAM.

def month_diff_expr_safe(month_col, prev_col):
    return (
        (pl.col(month_col).dt.year() - pl.col(prev_col).dt.year()) * 12
        + (pl.col(month_col).dt.month() - pl.col(prev_col).dt.month())
    )


def add_features_pl(df: pl.DataFrame) -> pl.DataFrame:
    timer("Start SIMPLE feature engineering with Polars")
    keys = ["location", "item_id"]

    # ============================================================
    # Normalize month dtype
    # ============================================================
    if df.schema["month"] == pl.Utf8:
        df = df.with_columns(
            pl.col("month")
            .str.strptime(pl.Date, "%Y-%m-%d", strict=False)
            .alias("month")
        )
    elif df.schema["month"] == pl.Datetime:
        df = df.with_columns(pl.col("month").dt.date().alias("month"))
    elif df.schema["month"] != pl.Date:
        df = df.with_columns(
            pl.col("month").cast(pl.Date, strict=False).alias("month")
        )

    df = df.with_columns(
        pl.col(TARGET).cast(pl.Float64, strict=False).alias(TARGET)
    )

    df = df.sort(keys + ["month"])

    # ============================================================
    # Calendar
    # ============================================================
    month_num_expr = pl.col("month").dt.month()

    df = df.with_columns([
        month_num_expr.cast(pl.Int16).alias("month_num"),
        pl.col("month").dt.quarter().cast(pl.Int16).alias("quarter_of_year"),
        (month_num_expr == 12).cast(pl.Int8).alias("is_christmas_month"),
        month_num_expr.is_in([1, 2]).cast(pl.Int8).alias("is_tet_month"),
        month_num_expr.is_in([9, 10, 11, 12]).cast(pl.Int8).alias("is_big_sale_month"),
        month_num_expr.is_in([6, 7, 8]).cast(pl.Int8).alias("is_summer_month"),
    ])

    # ============================================================
    # Pair sequence/gap features
    # ============================================================
    df = (
        df.with_columns([
            pl.col("month").shift(1).over(keys).alias("prev_month"),
            (pl.col("month").cum_count().over(keys) - 1)
            .cast(pl.Int16)
            .alias("n_rows_before"),
        ])
        .with_columns([
            month_diff_expr_safe("month", "prev_month")
            .fill_null(99)
            .clip(0, 99)
            .cast(pl.Int16)
            .alias("gap_from_prev_month")
        ])
    )

    # ============================================================
    # Quantity lags
    # ============================================================
    df = (
        df.with_columns([
            pl.col(TARGET).shift(1).over(keys).alias("qty_lag_1"),
            pl.col(TARGET).shift(2).over(keys).alias("qty_lag_2"),
            pl.col(TARGET).shift(3).over(keys).alias("qty_lag_3"),
        ])
        .with_columns([
            pl.mean_horizontal([
                pl.col("qty_lag_1"),
                pl.col("qty_lag_2"),
                pl.col("qty_lag_3"),
            ]).alias("qty_roll_mean_3"),

            pl.max_horizontal([
                pl.col("qty_lag_1"),
                pl.col("qty_lag_2"),
                pl.col("qty_lag_3"),
            ]).alias("qty_roll_max_3"),
        ])
    )

    # ============================================================
    # Sale occurrence lags
    # ============================================================
    df = df.with_columns(
        (pl.col(TARGET).fill_null(0) > 0)
        .cast(pl.Int8)
        .alias("sold_flag")
    )

    df = (
        df.with_columns([
            pl.col("sold_flag").shift(1).over(keys).alias("sold_lag_1"),
            pl.col("sold_flag").shift(2).over(keys).alias("sold_lag_2"),
            pl.col("sold_flag").shift(3).over(keys).alias("sold_lag_3"),
        ])
        .with_columns([
            pl.sum_horizontal([
                pl.col("sold_lag_1").fill_null(0),
                pl.col("sold_lag_2").fill_null(0),
                pl.col("sold_lag_3").fill_null(0),
            ]).alias("sold_roll_sum_3")
        ])
    )

    # Last sale gap: vì panel simple chỉ có observed rows + forecast,
    # gap_from_prev_month là proxy nhẹ, ít RAM.
    df = df.with_columns(
        pl.col("gap_from_prev_month")
        .clip(0, 12)
        .cast(pl.Int16)
        .alias("months_since_last_sale")
    )

    # ============================================================
    # Chỉ lấy lag_1 cho vài cột event/price
    # ============================================================
    maybe_lag1_cols = [
        "purchase_count",
        "view_item_count",
        "add_to_cart_count",
        "discount_sum",
        "avg_txn_price",
        "price",
    ]

    lag_exprs = []
    for c in maybe_lag1_cols:
        if c in df.columns:
            lag_exprs.append(
                pl.col(c)
                .cast(pl.Float64, strict=False)
                .shift(1)
                .over(keys)
                .fill_null(0.0)
                .cast(pl.Float32)
                .alias(f"{c}_lag_1")
            )

    if lag_exprs:
        df = df.with_columns(lag_exprs)

    # ============================================================
    # Label codes
    # ============================================================
    code_cols = [
        "location",
        "item_id",
        "cat_1",
        "cat_2",
        "cat_3",
        "cat",
        "category",
        "brand",
        "demand_cluster",
    ]

    code_exprs = []
    for c in code_cols:
        if c in df.columns:
            code_exprs.append(
                pl.col(c)
                .cast(pl.Utf8, strict=False)
                .cast(pl.Categorical)
                .to_physical()
                .cast(pl.Int32)
                .fill_null(-1)
                .alias(f"{c}_code")
            )

    if code_exprs:
        df = df.with_columns(code_exprs)

    df = df.drop(["prev_month"], strict=False)

    # ============================================================
    # Fill numeric feature null/inf only
    # TARGET giữ null cho forecast.
    #
    # FIX quan trọng:
    # - Float columns: xử lý inf/nan/null.
    # - Integer columns: chỉ fill_null, không replace inf.
    # Vì Int16/Int32 không nhận được inf/-inf.
    # ============================================================
    numeric_cols = df.select(cs.numeric()).columns

    float_dtypes = {pl.Float32, pl.Float64}

    fill_exprs = []
    for c in numeric_cols:
        if c == TARGET:
            continue

        dtype = df.schema[c]

        if dtype in float_dtypes:
            fill_exprs.append(
                pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan())
                .then(None)
                .otherwise(pl.col(c))
                .fill_null(0.0)
                .cast(pl.Float32)
                .alias(c)
            )
        else:
            fill_exprs.append(
                pl.col(c)
                .fill_null(0)
                .alias(c)
            )

    if fill_exprs:
        df = df.with_columns(fill_exprs)

    timer("SIMPLE feature engineering done")
    return df


# Đổi cache name để không load lại cache lỗi cũ
panel_feat_cache_name = f"panel_feat_SIMPLE_LOW_RAM_{zero_tag}_{CLUSTER_VERSION}_FIXED.parquet"

panel_feat_pl = maybe_load_pl(panel_feat_cache_name)

if panel_feat_pl is None:
    panel_feat_pl = add_features_pl(panel_smart)
    maybe_save_pl(panel_feat_pl, panel_feat_cache_name)

print("panel_feat_pl shape:", panel_feat_pl.shape)
display(panel_feat_pl.head())
display(panel_feat_pl.tail())

gc.collect()

[16:28:02] Start SIMPLE feature engineering with Polars
[16:29:35] SIMPLE feature engineering done
[16:29:54] Saved Polars cache: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/panel_feat_SIMPLE_LOW_RAM_SMART_ZERO_intermittent_low_volume_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_POLARS_LGBM_XGB_FIXED.parquet
panel_feat_pl shape: (17884069, 60)


month,location,item_id,sales_qty,purchase_count,unique_bills,unique_customers,avg_txn_price,discount_sum,discount_mean,is_observed_sale_month,is_smart_zero,is_forecast,view_item_count,add_to_cart_count,price,first_month,last_month,active_months,total_qty,mean_qty,median_qty,max_qty,history_months,months_since_last_sale_at_cutoff,density,demand_cluster,category,brand,item_price,month_num,quarter_of_year,is_christmas_month,is_tet_month,is_big_sale_month,is_summer_month,n_rows_before,gap_from_prev_month,qty_lag_1,qty_lag_2,qty_lag_3,qty_roll_mean_3,qty_roll_max_3,sold_flag,sold_lag_1,sold_lag_2,sold_lag_3,sold_roll_sum_3,months_since_last_sale,purchase_count_lag_1,view_item_count_lag_1,add_to_cart_count_lag_1,discount_sum_lag_1,avg_txn_price_lag_1,price_lag_1,location_code,item_id_code,category_code,brand_code,demand_cluster_code
date,str,str,f64,f32,f32,f32,f32,f32,f32,i8,i8,i8,f32,f32,f32,datetime[μs],datetime[μs],i16,f32,f32,f32,f32,i16,i16,f32,str,str,str,"decimal[38,4]",i16,i16,i8,i8,i8,i8,i16,i16,f32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i16,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,i32
2025-01-01,"""1000""","""0000280000138""",1.0,1.0,1.0,1.0,58500.0,6500.0,6500.0,1,0,0,0.0,0.0,58500.0,2025-01-01 00:00:00,null,0,0.0,0.0,0.0,0.0,0,0,0.0,"""cold_start""","""Đồ chơi nhà tắm""","""Tuyết Mai""",65000.0000,1,1,0,1,0,0,0,99,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,12,0.0,0.0,0.0,0.0,0.0,0.0,1,195,0,8,18392
2025-02-01,"""1000""","""0000280000138""",6.0,6.0,6.0,5.0,61701.449219,19791.3125,3298.552246,1,0,0,0.0,0.0,61701.449219,2025-01-01 00:00:00,2025-01-01 00:00:00,1,1.0,1.0,1.0,1.0,1,0,1.0,"""cold_start""","""Đồ chơi nhà tắm""","""Tuyết Mai""",65000.0000,2,1,0,1,0,0,1,1,1.0,0.0,0.0,1.0,1.0,1,1,0,0,1,1,1.0,0.0,0.0,6500.0,58500.0,58500.0,1,195,0,8,18392
2025-03-01,"""1000""","""0000280000138""",4.0,4.0,4.0,3.0,65000.0,0.0,0.0,1,0,0,0.0,0.0,65000.0,2025-01-01 00:00:00,2025-02-01 00:00:00,2,7.0,3.5,3.5,6.0,2,0,1.0,"""cold_start""","""Đồ chơi nhà tắm""","""Tuyết Mai""",65000.0000,3,1,0,0,0,0,2,1,6.0,1.0,0.0,3.5,6.0,1,1,1,0,2,1,6.0,0.0,0.0,19791.3125,61701.449219,61701.449219,1,195,0,8,18392
2025-04-01,"""1000""","""0000280000138""",6.0,6.0,6.0,6.0,59150.695312,35095.835938,5849.306152,1,0,0,0.0,0.0,59150.695312,2025-01-01 00:00:00,2025-03-01 00:00:00,3,11.0,3.666667,4.0,6.0,3,0,1.0,"""regular""","""Đồ chơi nhà tắm""","""Tuyết Mai""",65000.0000,4,2,0,0,0,0,3,1,4.0,6.0,1.0,3.666667,6.0,1,1,1,1,3,1,4.0,0.0,0.0,0.0,65000.0,65000.0,1,195,0,8,18393
2025-05-01,"""1000""","""0000280000138""",6.0,6.0,6.0,5.0,59583.332031,32500.0,5416.666504,1,0,0,0.0,0.0,59583.332031,2025-01-01 00:00:00,2025-04-01 00:00:00,4,17.0,4.25,5.0,6.0,4,0,1.0,"""regular""","""Đồ chơi nhà tắm""","""Tuyết Mai""",65000.0000,5,2,0,0,0,0,4,1,6.0,4.0,6.0,5.333333,6.0,1,1,1,1,3,1,6.0,0.0,0.0,35095.835938,59150.695312,59150.695312,1,195,0,8,18393


month,location,item_id,sales_qty,purchase_count,unique_bills,unique_customers,avg_txn_price,discount_sum,discount_mean,is_observed_sale_month,is_smart_zero,is_forecast,view_item_count,add_to_cart_count,price,first_month,last_month,active_months,total_qty,mean_qty,median_qty,max_qty,history_months,months_since_last_sale_at_cutoff,density,demand_cluster,category,brand,item_price,month_num,quarter_of_year,is_christmas_month,is_tet_month,is_big_sale_month,is_summer_month,n_rows_before,gap_from_prev_month,qty_lag_1,qty_lag_2,qty_lag_3,qty_roll_mean_3,qty_roll_max_3,sold_flag,sold_lag_1,sold_lag_2,sold_lag_3,sold_roll_sum_3,months_since_last_sale,purchase_count_lag_1,view_item_count_lag_1,add_to_cart_count_lag_1,discount_sum_lag_1,avg_txn_price_lag_1,price_lag_1,location_code,item_id_code,category_code,brand_code,demand_cluster_code
date,str,str,f64,f32,f32,f32,f32,f32,f32,i8,i8,i8,f32,f32,f32,datetime[μs],datetime[μs],i16,f32,f32,f32,f32,i16,i16,f32,str,str,str,"decimal[38,4]",i16,i16,i8,i8,i8,i8,i16,i16,f32,f32,f32,f32,f32,i8,i8,i8,i8,i8,i16,f32,f32,f32,f32,f32,f32,i32,i32,i32,i32,i32
2025-11-01,"""999""","""7519000000004""",2.0,2.0,2.0,2.0,251443.0,55114.0,27557.0,1,0,0,0.0,0.0,251443.0,2025-11-01 00:00:00,null,0,0.0,0.0,0.0,0.0,0,0,0.0,"""cold_start""","""Oxygen_Tã Quần""","""Molfix""",279000.0000,11,4,0,0,1,0,0,99,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,12,0.0,0.0,0.0,0.0,0.0,0.0,17654,5107,1496,1959,18392
2025-12-01,"""999""","""7519000000004""",1.0,1.0,1.0,1.0,269000.0,10000.0,10000.0,1,0,0,0.0,0.0,269000.0,2025-11-01 00:00:00,2025-11-01 00:00:00,1,2.0,2.0,2.0,2.0,1,0,1.0,"""cold_start""","""Oxygen_Tã Quần""","""Molfix""",279000.0000,12,4,1,0,1,0,1,1,2.0,0.0,0.0,2.0,2.0,1,1,0,0,1,1,2.0,0.0,0.0,55114.0,251443.0,251443.0,17654,5107,1496,1959,18392
2026-01-01,"""999""","""7519000000004""",null,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0.0,0.0,0.0,2025-11-01 00:00:00,null,0,0.0,0.0,0.0,0.0,0,0,0.0,"""cold_start""","""Oxygen_Tã Quần""","""Molfix""",279000.0000,1,1,0,1,0,0,2,1,1.0,2.0,0.0,1.5,2.0,0,1,1,0,2,1,1.0,0.0,0.0,10000.0,269000.0,269000.0,17654,5107,1496,1959,18392
2025-12-01,"""999""","""7526000000001""",3.0,3.0,3.0,2.0,115000.0,60000.0,20000.0,1,0,0,0.0,0.0,115000.0,2025-12-01 00:00:00,null,0,0.0,0.0,0.0,0.0,0,0,0.0,"""cold_start""","""Kem đặc trị Ích Nhi""","""Ích Nhi""",135000.0000,12,4,1,0,1,0,0,99,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0,12,0.0,0.0,0.0,0.0,0.0,0.0,17654,8131,2273,3745,18392
2026-01-01,"""999""","""7526000000001""",null,0.0,0.0,0.0,0.0,0.0,0.0,0,0,1,0.0,0.0,0.0,2025-12-01 00:00:00,null,0,0.0,0.0,0.0,0.0,0,0,0.0,"""cold_start""","""Kem đặc trị Ích Nhi""","""Ích Nhi""",135000.0000,1,1,0,1,0,0,1,1,3.0,0.0,0.0,3.0,3.0,0,1,0,0,1,1,3.0,0.0,0.0,60000.0,115000.0,115000.0,17654,8131,2273,3745,18392


0

## 8. Split train/valid/test theo thời gian

Train có thể chứa smart-zero rows. Valid/test chỉ đánh giá trên tháng có sale thật để MAPE không bị undefined ở `y=0`.


In [8]:
# ============================================================
# 8. TIME SPLIT - TRAIN JAN-NOV, VALID DEC, FORECAST JAN - POLARS SELECT THEN PANDAS
# ============================================================
# Polars xử lý/split trước, chỉ convert sang pandas các cột cần cho LightGBM/evaluation.

assert TRAIN_START < VALID_MONTH < FORECAST_MONTH, (
    f"Invalid split order: TRAIN_START={TRAIN_START}, VALID_MONTH={VALID_MONTH}, FORECAST_MONTH={FORECAST_MONTH}"
)

def build_no_leak_feature_list_pl(df: pl.DataFrame) -> list:
    exclude_cols = {
        TARGET, "month", "location", "item_id",
        "first_month", "last_month", "prev_sale_month", "sale_gap",
        "is_forecast", "is_observed_sale_month", "is_smart_zero",
        "sold_flag", "last_sale_month_tmp", "last_sale_month_before",
        "status", "status_code", "item_price",
    }
    raw_current_signal_cols = {
        "purchase_count", "unique_bills", "unique_customers",
        "view_item_count", "add_to_cart_count",
        "discount_sum", "discount_mean", "avg_txn_price", "price",
    }
    leak_like = {
        "total_qty", "mean_qty", "median_qty", "max_qty", "active_months", "density",
        "spike_ratio", "span_months", "history_months", "avg_gap_months",
        "months_since_last_sale_at_cutoff",
        "item_mean_lag_safe", "location_mean_lag_safe", "brand_mean_lag_safe",
        "cat3_mean_lag_safe", "location_item_mean_lag_safe",
    }

    numeric_cols = df.select(cs.numeric()).columns
    features = []
    for c in numeric_cols:
        if c in exclude_cols or c in raw_current_signal_cols or c in leak_like:
            continue
        if "_mean_lag_safe" in c:
            continue
        if c.endswith("_current") or c.endswith("_latest"):
            continue
        features.append(c)

    bad = sorted(set(features) & (exclude_cols | raw_current_signal_cols | leak_like))
    assert not bad, f"Leak-prone columns found in FEATURES: {bad}"
    return features

FEATURES = build_no_leak_feature_list_pl(panel_feat_pl)

meta_cols = [
    "month", "location", "item_id", TARGET,
    "demand_cluster", "is_forecast", "is_observed_sale_month", "is_smart_zero",
]
fallback_cat_cols = ["cat_1", "cat_2", "cat_3", "cat", "category", "brand"]
SPLIT_KEEP_COLS = [c for c in dict.fromkeys(meta_cols + fallback_cat_cols + FEATURES) if c in panel_feat_pl.columns]

missing_required = [c for c in ["month", "location", "item_id", TARGET, "demand_cluster", "is_forecast"] if c not in SPLIT_KEEP_COLS]
if missing_required:
    raise ValueError(f"Thiếu cột bắt buộc sau feature selection: {missing_required}")

base_observed = (pl.col("is_forecast") == 0) & pl.col(TARGET).is_not_null()
train_filter = base_observed & (pl.col("month") >= month_lit(TRAIN_START)) & (pl.col("month") < month_lit(VALID_MONTH))
valid_filter = base_observed & (pl.col("is_observed_sale_month") == 1) & (pl.col("month") >= month_lit(VALID_MONTH)) & (pl.col("month") < month_lit(next_month(VALID_MONTH)))
forecast_filter = (pl.col("is_forecast") == 1) & (pl.col("month") >= month_lit(FORECAST_MONTH)) & (pl.col("month") < month_lit(next_month(FORECAST_MONTH)))

train_full_rows = int(panel_feat_pl.lazy().filter(train_filter).select(pl.len()).collect().item())
valid_rows = int(panel_feat_pl.lazy().filter(valid_filter).select(pl.len()).collect().item())
forecast_rows = int(panel_feat_pl.lazy().filter(forecast_filter).select(pl.len()).collect().item())

train_pl = panel_feat_pl.lazy().filter(train_filter).select(SPLIT_KEEP_COLS).collect()
if MAX_TRAIN_ROWS is not None and train_pl.height > MAX_TRAIN_ROWS:
    # Sample sau khi đã select ít cột để giảm RAM; vẫn stratify theo cluster bằng pandas helper.
    tmp_pd = train_pl.to_pandas()
    idx = sample_index_by_cluster(tmp_pd, pd.Series(True, index=tmp_pd.index), MAX_TRAIN_ROWS, RANDOM_STATE)
    train_df = tmp_pd.loc[idx].copy().reset_index(drop=True)
else:
    train_df = train_pl.to_pandas()

valid_tune_df = panel_feat_pl.lazy().filter(valid_filter).select(SPLIT_KEEP_COLS).collect().to_pandas()
forecast_df = panel_feat_pl.lazy().filter(forecast_filter).select(SPLIT_KEEP_COLS).collect().to_pandas()

if "is_observed_sale_month" in valid_tune_df.columns:
    valid_observed_df = valid_tune_df.loc[valid_tune_df["is_observed_sale_month"].eq(1)].copy().reset_index(drop=True)
else:
    valid_observed_df = valid_tune_df.iloc[0:0].copy()

# Backward-compatible names.
valid_df = valid_tune_df
test_df = valid_tune_df.iloc[0:0].copy()
test_eval_df = test_df
test_observed_df = test_df

print("Split months:")
print("TRAIN_START:", TRAIN_START.strftime("%Y-%m"))
print("TRAIN_END_INCLUSIVE: 2025-11")
print("VALID_MONTH:", VALID_MONTH.strftime("%Y-%m"))
print("FORECAST_MONTH:", FORECAST_MONTH.strftime("%Y-%m"))
print("n_features:", len(FEATURES))
print("n_cols copied per split:", len(SPLIT_KEEP_COLS), "/ original panel cols:", panel_feat_pl.width)
print("train_full rows before optional sample:", f"{train_full_rows:,}")
print("train_df used:", train_df.shape)
if "is_smart_zero" in train_df.columns:
    print("train smart-zero rows used:", f"{int(train_df['is_smart_zero'].sum()):,}")
print("valid_tune_df Dec rows:", valid_tune_df.shape, "raw count:", f"{valid_rows:,}")
print("valid_observed_df Dec diagnostic:", valid_observed_df.shape)
print("forecast_df Jan rows with features:", forecast_df.shape, "raw count:", f"{forecast_rows:,}")

empty_splits = []
for name, n_rows in {"train_df_full": train_full_rows, "valid_tune_df": len(valid_tune_df), "forecast_df": len(forecast_df)}.items():
    if n_rows == 0:
        empty_splits.append(name)
if empty_splits:
    print("WARNING: empty split(s):", empty_splits)
    mm = panel_feat_pl.select([pl.col("month").min().alias("min_month"), pl.col("month").max().alias("max_month")])
    print("Available panel month range:")
    display(mm)

print("Train cluster counts:")
display(train_df["demand_cluster"].value_counts(dropna=False))
print("Valid Dec cluster counts:")
display(valid_df["demand_cluster"].value_counts(dropna=False))
print("Feature leakage audit passed.")

del train_pl
gc.collect()


Split months:
TRAIN_START: 2025-01
TRAIN_END_INCLUSIVE: 2025-11
VALID_MONTH: 2025-12
FORECAST_MONTH: 2026-01
n_features: 29
n_cols copied per split: 39 / original panel cols: 60
train_full rows before optional sample: 13,026,696
train_df used: (800000, 39)
train smart-zero rows used: 125,353
valid_tune_df Dec rows: (1185669, 39) raw count: 1,185,669
valid_observed_df Dec diagnostic: (1185669, 39)
forecast_df Jan rows with features: (3144558, 39) raw count: 3,144,558
Train cluster counts:


demand_cluster
cold_start      382702
regular         178236
intermittent    122257
low_volume      116805
Name: count, dtype: int64

Valid Dec cluster counts:


demand_cluster
regular         438640
cold_start      423623
low_volume      177767
intermittent    145639
Name: count, dtype: int64

Feature leakage audit passed.


0

## 9. Feature list

In [9]:
# ============================================================
# 9. FEATURE LIST AUDIT
# ============================================================
# FEATURES đã được build ở cell split bằng Polars schema để tránh convert full panel sang pandas.
print("n_features:", len(FEATURES))
print("First 120 FEATURES:")
print(FEATURES[:120])

required_feature_examples = [
    "qty_lag_1", "qty_roll_mean_3", "sold_lag_1", "sold_roll_sum_3",
    "months_since_last_sale", "location_code", "item_id_code", "demand_cluster_code"
]
missing_examples = [c for c in required_feature_examples if c not in FEATURES]
if missing_examples:
    print("WARNING missing expected feature examples:", missing_examples)
else:
    print("Feature audit passed: key lag/sale/code features exist.")


n_features: 29
First 120 FEATURES:
['month_num', 'quarter_of_year', 'is_christmas_month', 'is_tet_month', 'is_big_sale_month', 'is_summer_month', 'n_rows_before', 'gap_from_prev_month', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_roll_mean_3', 'qty_roll_max_3', 'sold_lag_1', 'sold_lag_2', 'sold_lag_3', 'sold_roll_sum_3', 'months_since_last_sale', 'purchase_count_lag_1', 'view_item_count_lag_1', 'add_to_cart_count_lag_1', 'discount_sum_lag_1', 'avg_txn_price_lag_1', 'price_lag_1', 'location_code', 'item_id_code', 'category_code', 'brand_code', 'demand_cluster_code']
Feature audit passed: key lag/sale/code features exist.


## 10. Model helpers: LightGBM + XGBoost + integer MAPE rounder + candidate selector


In [10]:
# ============================================================
# 10. SIMPLE MODEL HELPERS: LightGBM + XGBoost + candidate selector
# ============================================================
import gc
import numpy as np
import pandas as pd


def get_clip_max(cluster):
    if cluster == "cold_start":
        return 1
    if cluster == "low_volume":
        return 6
    if cluster == "intermittent":
        return 10
    return None


def fallback_predict(train_part: pd.DataFrame, data: pd.DataFrame) -> np.ndarray:
    """Median fallback trên positive rows."""
    pos = train_part[train_part[TARGET] > 0].copy()
    if len(pos) == 0:
        return np.ones(len(data), dtype=float)

    global_med = float(pos[TARGET].median())
    item_med = pos.groupby("item_id", observed=True)[TARGET].median()
    loc_med = pos.groupby("location", observed=True)[TARGET].median()

    pred = data["item_id"].map(item_med)
    for cat_col in ["cat_3", "cat_2", "cat_1", "cat", "category", "brand"]:
        if cat_col in pos.columns and cat_col in data.columns:
            cat_med = pos.groupby(cat_col, observed=True)[TARGET].median()
            pred = pred.fillna(data[cat_col].map(cat_med))
    pred = pred.fillna(data["location"].map(loc_med)).fillna(global_med)
    return pred.astype(float).values


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------
def get_lgbm_params_by_cluster(cluster: str) -> dict:
    common = {
        "objective": "regression_l1",
        "metric": "l1",
        "learning_rate": 0.07,
        "force_col_wise": True,
        "max_bin": 63,
        "num_threads": -1,
        "verbosity": -1,
        "seed": RANDOM_STATE,
        "feature_pre_filter": False,
    }
    if cluster == "low_volume":
        return {**common, "num_leaves": 5, "min_data_in_leaf": 250, "feature_fraction": 0.70, "bagging_fraction": 0.70, "bagging_freq": 1, "reg_alpha": 2.0, "reg_lambda": 10.0}
    if cluster == "intermittent":
        return {**common, "num_leaves": 7, "min_data_in_leaf": 220, "feature_fraction": 0.72, "bagging_fraction": 0.72, "bagging_freq": 1, "reg_alpha": 1.8, "reg_lambda": 12.0}
    return {**common, "num_leaves": 11, "min_data_in_leaf": 180, "feature_fraction": 0.75, "bagging_fraction": 0.75, "bagging_freq": 1, "reg_alpha": 1.0, "reg_lambda": 8.0}


def train_lgbm_regressor(tr, va=None, cluster="regular"):
    if not HAS_LIGHTGBM:
        return None

    params = get_lgbm_params_by_cluster(cluster)
    X_tr = tr[FEATURES]
    y_tr = tr[TARGET].astype(float).values
    w_tr = 1.0 / np.maximum(y_tr, 1.0)  # Weighted L1 gần MAPE
    if "is_smart_zero" in tr.columns:
        zero_mask = tr["is_smart_zero"].astype(int).values == 1
        w_tr[zero_mask] = float(globals().get("ZERO_SAMPLE_WEIGHT", 0.25))

    ds_tr = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, free_raw_data=True)
    valid_sets = [ds_tr]
    valid_names = ["train"]
    callbacks = [lgb.log_evaluation(100)]

    if va is not None and len(va) >= 100:
        X_va = va[FEATURES]
        y_va = va[TARGET].astype(float).values
        w_va = 1.0 / np.maximum(y_va, 1.0)
        if "is_smart_zero" in va.columns:
            zero_mask_va = va["is_smart_zero"].astype(int).values == 1
            w_va[zero_mask_va] = float(globals().get("ZERO_SAMPLE_WEIGHT", 0.25))
        ds_va = lgb.Dataset(X_va, label=y_va, weight=w_va, reference=ds_tr, free_raw_data=True)
        valid_sets.append(ds_va)
        valid_names.append("valid")
        callbacks.append(lgb.early_stopping(stopping_rounds=50, first_metric_only=True, verbose=False))

    if cluster == "low_volume":
        num_rounds = 1000
    elif cluster == "intermittent":
        num_rounds = 1000
    else:
        num_rounds = 1000

    model = lgb.train(
        params,
        ds_tr,
        num_boost_round=num_rounds,
        valid_sets=valid_sets,
        valid_names=valid_names,
        callbacks=callbacks,
    )
    return model


# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------
def get_xgb_params_by_cluster(cluster: str) -> dict:
    # Dùng tree_method=hist để chạy nhẹ hơn trên Kaggle CPU.
    common = {
        "objective": "reg:squarederror",
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_child_weight": 80,
        "subsample": 0.75,
        "colsample_bytree": 0.75,
        "reg_alpha": 1.0,
        "reg_lambda": 8.0,
        "n_estimators": 1000,
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
        "verbosity": 0,
    }
    if cluster == "low_volume":
        return {**common, "max_depth": 3, "min_child_weight": 160, "n_estimators": 1000, "reg_alpha": 2.0, "reg_lambda": 12.0}
    if cluster == "intermittent":
        return {**common, "max_depth": 3, "min_child_weight": 140, "n_estimators": 1000, "reg_alpha": 2.0, "reg_lambda": 14.0}
    return common


def train_xgb_regressor(tr, va=None, cluster="regular"):
    if not HAS_XGBOOST:
        return None

    params = get_xgb_params_by_cluster(cluster)
    X_tr = tr[FEATURES]
    y_tr = tr[TARGET].astype(float).values
    w_tr = 1.0 / np.maximum(y_tr, 1.0)
    if "is_smart_zero" in tr.columns:
        zero_mask = tr["is_smart_zero"].astype(int).values == 1
        w_tr[zero_mask] = float(globals().get("ZERO_SAMPLE_WEIGHT", 0.25))

    model = xgb.XGBRegressor(**params)

    # XGBoost version ở Kaggle có thể khác nhau, nên fit theo try/except.
    if va is not None and len(va) >= 100:
        X_va = va[FEATURES]
        y_va = va[TARGET].astype(float).values
        w_va = 1.0 / np.maximum(y_va, 1.0)
        if "is_smart_zero" in va.columns:
            zero_mask_va = va["is_smart_zero"].astype(int).values == 1
            w_va[zero_mask_va] = float(globals().get("ZERO_SAMPLE_WEIGHT", 0.25))
        try:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=w_tr,
                eval_set=[(X_va, y_va)],
                sample_weight_eval_set=[w_va],
                verbose=False,
            )
        except TypeError:
            model.fit(X_tr, y_tr, sample_weight=w_tr, verbose=False)
    else:
        try:
            model.fit(X_tr, y_tr, sample_weight=w_tr, verbose=False)
        except TypeError:
            model.fit(X_tr, y_tr, sample_weight=w_tr)

    return model


# ------------------------------------------------------------
# Rounding + candidate search
# ------------------------------------------------------------
def custom_round_integer_mape(pred, t01=0.7, t12=1.6, clip_max=None, min_pred_int=1):
    pred = np.asarray(pred, dtype=float)
    pred = np.clip(pred, 0, None)
    if clip_max is not None:
        pred = np.clip(pred, 0, clip_max)
    pred_int = custom_round_low(pred, t01=t01, t12=t12)
    pred_int = np.asarray(pred_int, dtype=int)
    if min_pred_int is not None:
        pred_int = np.maximum(pred_int, int(min_pred_int))
    return pred_int


def optimize_rounding(y_true, pred, clip_max=None, allow_zero_pred=False, cluster="regular"):
    y_true = np.asarray(y_true, dtype=float)
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    min_pred_int = 0 if allow_zero_pred else 1

    if cluster == "low_volume":
        t01_list = [0.5, 0.6, 0.7]
        t12_list = [1.3, 1.5, 1.7]
        clip_list = [4, 6, None] if clip_max is None else [clip_max]
    elif cluster == "intermittent":
        t01_list = [0.5, 0.6, 0.7]
        t12_list = [1.3, 1.5, 1.7, 1.9]
        clip_list = [8, 10, None] if clip_max is None else [clip_max]
    else:
        t01_list = [0.5, 0.6, 0.7]
        t12_list = [1.5, 1.7]
        clip_list = [None] if clip_max is None else [clip_max]

    best = {
        "method": "model_integer",
        "MAPE": np.inf,
        "t01": 0.6,
        "t12": 1.5,
        "clip_max": clip_max,
        "min_pred_int": min_pred_int,
        "eval_pred_type": "integer",
    }

    for cm in clip_list:
        for t01 in t01_list:
            for t12 in t12_list:
                if t12 <= t01:
                    continue
                pi = custom_round_integer_mape(
                    pred,
                    t01=t01,
                    t12=t12,
                    clip_max=cm,
                    min_pred_int=min_pred_int,
                )
                score = mape_score(y_true, pi)
                if score < best["MAPE"]:
                    best = {
                        "method": "model_integer",
                        "MAPE": float(score),
                        "t01": float(t01),
                        "t12": float(t12),
                        "clip_max": None if cm is None else float(cm),
                        "min_pred_int": int(min_pred_int),
                        "eval_pred_type": "integer",
                    }
    return best


def predict_single_model(model, data: pd.DataFrame) -> np.ndarray:
    if model is None:
        raise ValueError("model is None")
    pred = model.predict(data[FEATURES])
    return np.clip(np.asarray(pred, dtype=float), 0, None)


def make_candidates(data: pd.DataFrame, model_bank=None, cluster=None) -> dict:
    """Tạo candidates gồm LightGBM, XGBoost, lag/rolling, và blend nhẹ."""
    candidates = {}

    if isinstance(model_bank, dict):
        if model_bank.get("lightgbm") is not None:
            candidates["lgbm"] = predict_single_model(model_bank["lightgbm"], data)
        if model_bank.get("xgboost") is not None:
            candidates["xgb"] = predict_single_model(model_bank["xgboost"], data)
    elif model_bank is not None:
        # Backward compatible nếu models[cluster] là 1 model đơn.
        candidates["model"] = predict_single_model(model_bank, data)

    for name, col in [
        ("lag_1", "qty_lag_1"),
        ("roll_mean_3", "qty_roll_mean_3"),
    ]:
        if col in data.columns:
            candidates[name] = data[col].fillna(0).astype(float).values

    # Blend model với rolling mean nếu có.
    for model_name in ["lgbm", "xgb", "model"]:
        if model_name in candidates and "roll_mean_3" in candidates:
            candidates[f"blend_07_{model_name}_03_roll3"] = 0.7 * candidates[model_name] + 0.3 * candidates["roll_mean_3"]
        if model_name in candidates and "lag_1" in candidates:
            candidates[f"blend_07_{model_name}_03_lag1"] = 0.7 * candidates[model_name] + 0.3 * candidates["lag_1"]

    if "lgbm" in candidates and "xgb" in candidates:
        candidates["blend_05_lgbm_05_xgb"] = 0.5 * candidates["lgbm"] + 0.5 * candidates["xgb"]

    for k in list(candidates):
        candidates[k] = np.clip(np.asarray(candidates[k], dtype=float), 0, None)

    return candidates


def choose_best_candidate(y_true, data, model_bank, cluster, clip_max=None):
    candidates = make_candidates(data, model_bank=model_bank, cluster=cluster)
    if len(candidates) == 0:
        candidates["constant_one"] = np.ones(len(data), dtype=float)

    best_name, best_rule, best_score, best_pred, best_int = None, None, np.inf, None, None
    print(f"Candidate search for {cluster}:")

    for name, pred in candidates.items():
        rule = optimize_rounding(
            y_true,
            pred,
            clip_max=clip_max,
            allow_zero_pred=False,
            cluster=cluster,
        )
        pi = custom_round_integer_mape(
            pred,
            t01=rule["t01"],
            t12=rule["t12"],
            clip_max=rule["clip_max"],
            min_pred_int=rule["min_pred_int"],
        )
        score = mape_score(y_true, pi)
        print(f"  {name}: MAPE={score:.4f}, rule={rule}")
        if score < best_score:
            best_name, best_rule, best_score, best_pred, best_int = name, rule, score, pred, pi

    best_rule = {
        **best_rule,
        "method": "candidate_integer",
        "candidate_name": best_name,
        "cluster": cluster,
    }
    return best_pred, best_int, best_rule


def train_model_bank(tr, va=None, cluster="regular") -> dict:
    """Train LightGBM và XGBoost nếu có. Candidate search sẽ chọn cái tốt hơn."""
    bank = {"lightgbm": None, "xgboost": None}

    if HAS_LIGHTGBM:
        print("Train LightGBM...")
        bank["lightgbm"] = train_lgbm_regressor(tr, va, cluster=cluster)
    else:
        print("Skip LightGBM: not installed/import failed.")

    if globals().get("USE_XGBOOST", True) and HAS_XGBOOST:
        print("Train XGBoost...")
        bank["xgboost"] = train_xgb_regressor(tr, va, cluster=cluster)
    else:
        print("Skip XGBoost: disabled or not installed/import failed.")

    if bank["lightgbm"] is None and bank["xgboost"] is None:
        return None

    return bank


def predict_with_rule(model_bank, tr_cluster, data, rule):
    method = rule.get("method", "model_integer")

    if method == "constant_one":
        return np.ones(len(data), dtype=float), np.ones(len(data), dtype=int)

    if method == "candidate_integer":
        candidates = make_candidates(data, model_bank=model_bank, cluster=rule.get("cluster"))
        name = rule.get("candidate_name", "lgbm")
        if name == "constant_one":
            pred_float = np.ones(len(data), dtype=float)
        elif name in candidates:
            pred_float = candidates[name]
        elif "lgbm" in candidates:
            pred_float = candidates["lgbm"]
        elif "xgb" in candidates:
            pred_float = candidates["xgb"]
        elif "model" in candidates:
            pred_float = candidates["model"]
        else:
            pred_float = fallback_predict(tr_cluster, data)

    elif method.startswith("fallback") or model_bank is None:
        pred_float = fallback_predict(tr_cluster, data)

    else:
        # Backward compatible: ưu tiên LightGBM nếu có, sau đó XGBoost.
        if isinstance(model_bank, dict):
            model = model_bank.get("lightgbm") or model_bank.get("xgboost")
            pred_float = predict_single_model(model, data) if model is not None else fallback_predict(tr_cluster, data)
        else:
            pred_float = predict_single_model(model_bank, data)

    pred_float = np.clip(np.asarray(pred_float, dtype=float), 0, None)
    pred_int = custom_round_integer_mape(
        pred_float,
        t01=rule.get("t01", 0.6),
        t12=rule.get("t12", 1.5),
        clip_max=rule.get("clip_max", None),
        min_pred_int=rule.get("min_pred_int", 1),
    )
    pred_int = np.asarray(pred_int, dtype=int)
    return pred_float, pred_int


## 11. Train per cluster + compare LightGBM vs XGBoost trên valid Dec


In [11]:
# ============================================================
# 11. TRAIN PER 4 SIMPLE CLUSTERS - LightGBM vs XGBoost
# ============================================================
import gc
import json
import numpy as np
import pandas as pd

models = {}
cluster_rules = {}
valid_parts = []

# ============================================================
# Helper: convert numpy types để json.dump không lỗi
# ============================================================
def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)
    if obj is None:
        return None
    try:
        if pd.isna(obj):
            return None
    except Exception:
        pass
    return obj


def safe_evaluate(df):
    if df is None or len(df) == 0:
        return {"n": 0, "MAPE": None, "MAE": None, "RMSE": None}
    return evaluate_predictions(df)


# ============================================================
# Check dữ liệu
# ============================================================
if "train_df" not in globals():
    raise ValueError("Chưa có train_df. Hãy chạy cell split trước.")
if "valid_df" not in globals():
    raise ValueError("Chưa có valid_df. Hãy chạy cell split trước.")
if len(train_df) == 0:
    raise ValueError("train_df đang rỗng.")
if len(valid_df) == 0:
    raise ValueError("valid_df đang rỗng.")
if "demand_cluster" not in train_df.columns:
    raise ValueError("train_df thiếu cột demand_cluster.")
if "demand_cluster" not in valid_df.columns:
    raise ValueError("valid_df thiếu cột demand_cluster.")
if TARGET not in train_df.columns or TARGET not in valid_df.columns:
    raise ValueError(f"Thiếu cột TARGET = {TARGET} trong train_df hoặc valid_df.")

# ============================================================
# 4 nhóm. Dynamic để tránh lỗi nếu cluster nào không xuất hiện.
# ============================================================
base_order = ["cold_start", "intermittent", "low_volume", "regular"]
existing_clusters = (
    pd.concat([train_df["demand_cluster"], valid_df["demand_cluster"]], ignore_index=True)
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)
CLUSTER_ORDER = [c for c in base_order if c in existing_clusters] + [c for c in existing_clusters if c not in base_order]

EXPECTED_4_CLUSTERS = ["cold_start", "intermittent", "low_volume", "regular"]
print("CLUSTER_ORDER:", CLUSTER_ORDER)
print("Expected 4 clusters:", EXPECTED_4_CLUSTERS)
missing_train_valid_clusters = [c for c in EXPECTED_4_CLUSTERS if c not in existing_clusters]
if missing_train_valid_clusters:
    print("WARNING: Train/valid chưa có cluster:", missing_train_valid_clusters)
print("HAS_LIGHTGBM:", HAS_LIGHTGBM)
print("HAS_XGBOOST:", HAS_XGBOOST, "| USE_XGBOOST:", globals().get("USE_XGBOOST", True))

# ============================================================
# Train / tune từng cluster
# ============================================================
for cluster in CLUSTER_ORDER:
    print("\n" + "=" * 80)
    print("CLUSTER:", cluster)

    tr = train_df[train_df["demand_cluster"].astype(str) == cluster].copy()
    va = valid_df[valid_df["demand_cluster"].astype(str) == cluster].copy()
    print("train rows:", len(tr), "valid/tune rows:", len(va))

    clip_max = get_clip_max(cluster)

    if len(va) == 0:
        print("No validation rows for this cluster. Skip eval.")
        models[cluster] = None
        cluster_rules[cluster] = {
            "method": "constant_one" if cluster == "cold_start" else "fallback_median_integer",
            "cluster": cluster,
            "MAPE": None,
            "t01": 0.6,
            "t12": 1.5,
            "clip_max": clip_max,
            "min_pred_int": 1,
        }
        del tr, va
        gc.collect()
        continue

    y_true_va = va[TARGET].astype(float).values

    # ------------------------------------------------------------
    # Cold start: không train model, gán 1
    # ------------------------------------------------------------
    if cluster == "cold_start":
        pred_float = np.ones(len(va), dtype=float)
        pred_int = np.ones(len(va), dtype=int)
        score = mape_score(y_true_va, pred_int)
        models[cluster] = None
        cluster_rules[cluster] = {
            "method": "constant_one",
            "cluster": cluster,
            "MAPE": float(score),
            "t01": None,
            "t12": None,
            "clip_max": 1,
            "min_pred_int": 1,
            "eval_pred_type": "integer",
        }

    # ------------------------------------------------------------
    # Nếu train ít hoặc không có cả LightGBM/XGBoost thì fallback median
    # ------------------------------------------------------------
    elif len(tr) < 100 or (not HAS_LIGHTGBM and (not HAS_XGBOOST or not globals().get("USE_XGBOOST", True))):
        print("Use fallback median.")
        pred_float_raw = fallback_predict(train_df, va)
        best = optimize_rounding(
            y_true_va,
            pred_float_raw,
            clip_max=clip_max,
            allow_zero_pred=False,
            cluster=cluster,
        )
        rule = {"method": "fallback_median_integer", "cluster": cluster, **best}
        pred_float, pred_int = predict_with_rule(None, train_df, va, rule)
        models[cluster] = None
        cluster_rules[cluster] = rule

    # ------------------------------------------------------------
    # Train LightGBM + XGBoost, rồi candidate search chọn tốt nhất
    # ------------------------------------------------------------
    else:
        model_bank = train_model_bank(tr, va, cluster=cluster)

        if model_bank is None:
            print("No model trained. Use fallback median.")
            pred_float_raw = fallback_predict(train_df, va)
            best = optimize_rounding(
                y_true_va,
                pred_float_raw,
                clip_max=clip_max,
                allow_zero_pred=False,
                cluster=cluster,
            )
            rule = {"method": "fallback_median_integer", "cluster": cluster, **best}
            pred_float, pred_int = predict_with_rule(None, train_df, va, rule)
            models[cluster] = None
            cluster_rules[cluster] = rule
        else:
            pred_float, pred_int, rule = choose_best_candidate(
                y_true_va,
                va,
                model_bank,
                cluster,
                clip_max=clip_max,
            )
            models[cluster] = model_bank
            cluster_rules[cluster] = rule

    # ============================================================
    # Save valid prediction part
    # ============================================================
    keep_cols = ["month", "location", "item_id", TARGET, "demand_cluster"]
    if "is_observed_sale_month" in va.columns:
        keep_cols.append("is_observed_sale_month")

    out = va[keep_cols].copy()
    if "is_observed_sale_month" not in out.columns:
        out["is_observed_sale_month"] = 1

    out["y_pred_float"] = np.asarray(pred_float, dtype=float)
    out["y_pred_int"] = np.asarray(pred_int, dtype=int)
    out["quantity"] = np.maximum(out["y_pred_int"].values, 1).astype(int)
    out["selected_candidate"] = cluster_rules[cluster].get("candidate_name", cluster_rules[cluster].get("method"))

    valid_parts.append(out)

    print("Selected rule:")
    print(cluster_rules[cluster])

    print("Cluster valid metrics:")
    display(pd.DataFrame([safe_evaluate(out)]))

    del tr, va, out
    gc.collect()

# ============================================================
# Overall valid result
# ============================================================
if len(valid_parts) > 0:
    valid_pred_df = pd.concat(valid_parts, ignore_index=True)
else:
    valid_pred_df = pd.DataFrame()

print("\nOVERALL VALID/TUNE - PANEL ROWS")
display(pd.DataFrame([safe_evaluate(valid_pred_df)]))

if len(valid_pred_df) > 0 and "is_observed_sale_month" in valid_pred_df.columns:
    valid_pred_observed_df = valid_pred_df[valid_pred_df["is_observed_sale_month"] == 1].copy()
else:
    valid_pred_observed_df = valid_pred_df.copy()

print("OVERALL VALID/TUNE - OBSERVED SALE ROWS ONLY")
display(pd.DataFrame([safe_evaluate(valid_pred_observed_df)]))

print("VALID/TUNE BY CLUSTER")
if len(valid_pred_df) > 0:
    by_cluster = (
        valid_pred_df
        .groupby("demand_cluster", group_keys=False)
        .apply(lambda x: pd.Series(safe_evaluate(x)))
        .reset_index()
    )
    display(by_cluster)
else:
    print("valid_pred_df rỗng.")

print("SELECTED CANDIDATE BY CLUSTER")
if cluster_rules:
    display(pd.DataFrame([
        {
            "demand_cluster": c,
            "method": r.get("method"),
            "candidate_name": r.get("candidate_name"),
            "MAPE": r.get("MAPE"),
            "clip_max": r.get("clip_max"),
            "t01": r.get("t01"),
            "t12": r.get("t12"),
        }
        for c, r in cluster_rules.items()
    ]))

# ============================================================
# Save rules
# ============================================================
rules_path = CACHE_DIR / "cluster_rules_4cluster_lgbm_xgb.json"
with open(rules_path, "w", encoding="utf-8") as f:
    json.dump(to_jsonable(cluster_rules), f, ensure_ascii=False, indent=2)

print("Saved rules:", rules_path)
gc.collect()


CLUSTER_ORDER: ['cold_start', 'intermittent', 'low_volume', 'regular']
Expected 4 clusters: ['cold_start', 'intermittent', 'low_volume', 'regular']
HAS_LIGHTGBM: True
HAS_XGBOOST: True | USE_XGBOOST: True

CLUSTER: cold_start
train rows: 382702 valid/tune rows: 423623
Selected rule:
{'method': 'constant_one', 'cluster': 'cold_start', 'MAPE': 23.68118824231626, 't01': None, 't12': None, 'clip_max': 1, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
Cluster valid metrics:


,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,423623,23.681188,55.878165,1.266452,0.627185,0.811639



CLUSTER: intermittent
train rows: 122257 valid/tune rows: 145639
Train LightGBM...
Train XGBoost...
Candidate search for intermittent:
  lgbm: MAPE=27.6405, rule={'method': 'model_integer', 'MAPE': 27.640483368237184, 't01': 0.5, 't12': 1.3, 'clip_max': 10.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  xgb: MAPE=27.7055, rule={'method': 'model_integer', 'MAPE': 27.705470564201573, 't01': 0.5, 't12': 1.9, 'clip_max': 10.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  lag_1: MAPE=45.2129, rule={'method': 'model_integer', 'MAPE': 45.21287154016998, 't01': 0.5, 't12': 1.3, 'clip_max': 10.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  roll_mean_3: MAPE=36.7118, rule={'method': 'model_integer', 'MAPE': 36.71176098811212, 't01': 0.5, 't12': 1.7, 'clip_max': 10.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  blend_07_lgbm_03_roll3: MAPE=28.0705, rule={'method': 'model_integer', 'MAPE': 28.07046976939256, 't01': 0.5, 't12': 1.9, 'clip_max': 10.0, 'min_pred_int': 1, 'eval_

,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,145639,27.501697,51.173419,1.080239,0.550629,0.790454



CLUSTER: low_volume
train rows: 116805 valid/tune rows: 177767
Train LightGBM...
Train XGBoost...
Candidate search for low_volume:
  lgbm: MAPE=34.9697, rule={'method': 'model_integer', 'MAPE': 34.96974951525419, 't01': 0.5, 't12': 1.3, 'clip_max': 6.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  xgb: MAPE=35.1422, rule={'method': 'model_integer', 'MAPE': 35.142235183763084, 't01': 0.5, 't12': 1.7, 'clip_max': 6.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  lag_1: MAPE=52.4852, rule={'method': 'model_integer', 'MAPE': 52.48524448864767, 't01': 0.5, 't12': 1.3, 'clip_max': 6.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  roll_mean_3: MAPE=40.7579, rule={'method': 'model_integer', 'MAPE': 40.75788737096687, 't01': 0.5, 't12': 1.7, 'clip_max': 6.0, 'min_pred_int': 1, 'eval_pred_type': 'integer'}
  blend_07_lgbm_03_roll3: MAPE=34.9756, rule={'method': 'model_integer', 'MAPE': 34.97560716932552, 't01': 0.5, 't12': 1.7, 'clip_max': 6.0, 'min_pred_int': 1, 'eval_pred_type'

...

,demand_cluster,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,cold_start,423623.0,23.681188,55.878165,1.266452,0.627185,0.811639
1,intermittent,145639.0,27.501697,51.173419,1.080239,0.550629,0.790454
2,low_volume,177767.0,34.962936,53.496063,1.155681,0.434344,0.706914
3,regular,438640.0,54.386984,60.780983,6.490785,0.150360,0.382446


SELECTED CANDIDATE BY CLUSTER


,demand_cluster,method,candidate_name,MAPE,clip_max,t01,t12
0,cold_start,constant_one,None,23.681188,1.0,NaN,NaN
1,intermittent,candidate_integer,blend_05_lgbm_05_xgb,27.501697,10.0,0.5,1.3
2,low_volume,candidate_integer,blend_05_lgbm_05_xgb,34.962936,6.0,0.5,1.3
3,regular,candidate_integer,lgbm,54.386984,NaN,0.5,1.7


Saved rules: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/cluster_rules_4cluster_lgbm_xgb.json


33

## 12. Đánh giá Dec/2025 và lưu model bundle LightGBM/XGBoost


In [12]:
# ============================================================
# 12. DEC/2025 VALID EVALUATION + SAVE MODEL BUNDLE
# Includes LightGBM/XGBoost model bank + selected candidate per cluster
# ============================================================
import gc
import json
import numpy as np
import pandas as pd


def predict_by_cluster(data, train_source, models, rules):
    parts = []

    if data is None or len(data) == 0:
        return pd.DataFrame()

    if "demand_cluster" not in data.columns:
        raise ValueError("data thiếu cột demand_cluster.")

    for cluster, part in data.groupby("demand_cluster", observed=True):
        model_bank = models.get(cluster)
        rule = rules.get(cluster, {
            "method": "constant_one" if cluster == "cold_start" else "fallback_median_integer",
            "t01": 0.6,
            "t12": 1.5,
            "min_pred_int": 1,
            "clip_max": get_clip_max(cluster),
            "cluster": cluster,
        })

        tr_cluster = train_source[train_source["demand_cluster"] == cluster]
        if len(tr_cluster) == 0:
            tr_cluster = train_source

        pf, pi = predict_with_rule(model_bank, tr_cluster, part, rule)

        out_cols = ["month", "location", "item_id", "demand_cluster"]
        if TARGET in part.columns:
            out_cols.append(TARGET)
        if "is_observed_sale_month" in part.columns:
            out_cols.append("is_observed_sale_month")

        out = part[out_cols].copy()
        if "is_observed_sale_month" not in out.columns:
            out["is_observed_sale_month"] = 1

        out["y_pred_float"] = np.asarray(pf, dtype=float)
        out["y_pred_int"] = np.asarray(pi, dtype=int)
        out["y_pred"] = out["y_pred_int"].astype(int)
        out["quantity"] = np.maximum(out["y_pred_int"].values, 1).astype(int)
        out["pred_source"] = "model_existing_pair"
        out["selected_candidate"] = rule.get("candidate_name", rule.get("method"))
        parts.append(out)

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


# ============================================================
# Recompute Dec validation predictions using Jan-Nov models
# ============================================================
dec_pred_df = predict_by_cluster(valid_df, train_df, models, cluster_rules)

print("\n===== DEC 2025 VALID METRICS - PANEL ROWS =====")
if len(dec_pred_df):
    display(pd.DataFrame([
        evaluate_predictions(dec_pred_df, y_col=TARGET, pred_col="y_pred_int")
    ]))
else:
    print("dec_pred_df rỗng.")


dec_pred_observed_df = (
    dec_pred_df[dec_pred_df["is_observed_sale_month"] == 1].copy()
    if len(dec_pred_df) and "is_observed_sale_month" in dec_pred_df.columns
    else dec_pred_df.copy()
)

print("\n===== DEC 2025 VALID METRICS - OBSERVED SALE ROWS ONLY =====")
if len(dec_pred_observed_df):
    display(pd.DataFrame([
        evaluate_predictions(dec_pred_observed_df, y_col=TARGET, pred_col="y_pred_int")
    ]))
else:
    print("dec_pred_observed_df rỗng.")


print("\n===== DEC 2025 VALID BY CLUSTER - PANEL ROWS =====")
if len(dec_pred_df):
    display(
        dec_pred_df
        .groupby("demand_cluster", observed=True)
        .apply(lambda x: pd.Series(evaluate_predictions(x, y_col=TARGET, pred_col="y_pred_int")))
        .reset_index()
        .sort_values("MAPE")
    )


print("\n===== DEC 2025 VALID BY CLUSTER - OBSERVED SALE ROWS ONLY =====")
if len(dec_pred_observed_df):
    display(
        dec_pred_observed_df
        .groupby("demand_cluster", observed=True)
        .apply(lambda x: pd.Series(evaluate_predictions(x, y_col=TARGET, pred_col="y_pred_int")))
        .reset_index()
        .sort_values("MAPE")
    )

print("\n===== SELECTED MODEL/CANDIDATE BY CLUSTER =====")
if cluster_rules:
    display(pd.DataFrame([
        {
            "demand_cluster": c,
            "method": r.get("method"),
            "candidate_name": r.get("candidate_name"),
            "MAPE": r.get("MAPE"),
            "clip_max": r.get("clip_max"),
        }
        for c, r in cluster_rules.items()
    ]))

# ============================================================
# Save model bundle
# ============================================================
MODEL_DIR = CACHE_DIR / "saved_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEC_PRED_PATH = MODEL_DIR / "dec_2025_valid_predictions_train_202501_202511.pkl"
MODEL_BUNDLE_PATH = MODEL_DIR / "simple_4cluster_wide_intermit_lgbm_xgb_train_202501_202511_valid_202512.joblib"

dec_pred_df.to_pickle(DEC_PRED_PATH)

try:
    import joblib
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "joblib"])
    import joblib

model_bundle = {
    "models": models,
    "cluster_rules": cluster_rules,
    "FEATURES": FEATURES,
    "TARGET": TARGET,
    "fallback_train_df": train_df,
    "train_start": str(TRAIN_START.date()) if hasattr(TRAIN_START, "date") else str(TRAIN_START),
    "train_end_inclusive": "2025-11-30",
    "valid_month": str(VALID_MONTH.date()) if hasattr(VALID_MONTH, "date") else str(VALID_MONTH),
    "forecast_month": str(FORECAST_MONTH.date()) if hasattr(FORECAST_MONTH, "date") else str(FORECAST_MONTH),
    "cluster_version": CLUSTER_VERSION,
    "has_lightgbm": bool(HAS_LIGHTGBM),
    "has_xgboost": bool(HAS_XGBOOST),
    "use_xgboost": bool(globals().get("USE_XGBOOST", True)),
    "source": "simple_4cluster_low_ram_lgbm_xgb_candidate_search",
}

joblib.dump(model_bundle, MODEL_BUNDLE_PATH)

print("Saved DEC valid predictions:", DEC_PRED_PATH)
print("Saved model bundle:", MODEL_BUNDLE_PATH)

gc.collect()



===== DEC 2025 VALID METRICS - PANEL ROWS =====


,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,1185669,37.201596,59.130265,3.159721,0.412467,0.634555



===== DEC 2025 VALID METRICS - OBSERVED SALE ROWS ONLY =====


,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,1185669,37.201596,59.130265,3.159721,0.412467,0.634555



===== DEC 2025 VALID BY CLUSTER - PANEL ROWS =====


...

,demand_cluster,rows,MAPE,WMAPE,MAE,ACC_exact,ACC_within_1
0,cold_start,423623.0,23.681188,55.878165,1.266452,0.627185,0.811639
1,intermittent,145639.0,27.501697,51.173419,1.080239,0.550629,0.790454
2,low_volume,177767.0,34.962936,53.496063,1.155681,0.434344,0.706914
3,regular,438640.0,54.386984,60.780983,6.490785,0.150360,0.382446



===== SELECTED MODEL/CANDIDATE BY CLUSTER =====


,demand_cluster,method,candidate_name,MAPE,clip_max
0,cold_start,constant_one,None,23.681188,1.0
1,intermittent,candidate_integer,blend_05_lgbm_05_xgb,27.501697,10.0
2,low_volume,candidate_integer,blend_05_lgbm_05_xgb,34.962936,6.0
3,regular,candidate_integer,lgbm,54.386984,NaN


Saved DEC valid predictions: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/saved_models/dec_2025_valid_predictions_train_202501_202511.pkl
Saved model bundle: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/saved_models/simple_4cluster_wide_intermit_lgbm_xgb_train_202501_202511_valid_202512.joblib


59

## 13. Predict Jan/2026 và export full location × all item

In [13]:
# ============================================================
# 13. PREDICT JAN/2026 + EXPORT FULL LOCATION x ALL ITEM SAFELY
# ============================================================
# - Model: dùng models/rules vừa train từ Jan-Nov theo logic ronaldo.
# - Jan existing pairs: predict bằng model cho forecast_df.
# - Full output: location xuất hiện trong transaction 2025 x tất cả item
#   trong item table + item từng xuất hiện trong transaction.
# - Cặp không có prediction/feature => quantity = 1.
# - Export CSV theo chunk để tránh crash 1000 x 30000.
# - PKL full mặc định tắt vì 30M rows rất dễ crash RAM.

import os
import shutil
import gc
from pathlib import Path

if pl is None:
    raise ImportError("Notebook cần polars để export full location x item an toàn. Hãy cài polars trước.")

PRED_OUTPUT_DIR = DATA_DIR / "cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601"
PRED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from pathlib import Path

PRED_OUTPUT_DIR = Path("/kaggle/working/cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601")
PRED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PRED_BATCH_SIZE = globals().get("PRED_BATCH_SIZE", 350_000)
CHUNK_LOCATIONS = globals().get("CHUNK_LOCATIONS", 50)

CREATE_FULL_PKL = True

SUBMISSION_CSV = PRED_OUTPUT_DIR / "submission_sale_forecasting_train_202501_202511_valid_202512_predict_202601.csv"
SUBMISSION_PKL = PRED_OUTPUT_DIR / "submission_sale_forecasting_train_202501_202511_valid_202512_predict_202601.pkl"
DEBUG_PARQUET_DIR = PRED_OUTPUT_DIR / "jan_2026_debug_parquet_parts"
JAN_EXISTING_PRED_PKL = PRED_OUTPUT_DIR / "jan_2026_existing_feature_pair_predictions.pkl"

# -----------------------------
# 1) Predict Jan existing feature pairs by cluster, in pandas batches
# -----------------------------
jan_existing_pred_parts = []

if len(forecast_df) == 0:
    print("WARNING: forecast_df rỗng. Final full output sẽ toàn quantity=1.")
else:
    for cluster, part in forecast_df.groupby("demand_cluster", observed=True):
        model = models.get(cluster)
        rule = cluster_rules.get(cluster, {
            "method": "constant_one" if cluster == "cold_start" else "fallback_median_integer",
            "t01": 0.6,
            "t12": 1.5,
            "min_pred_int": 1,
            "clip_max": get_clip_max(cluster),
            "cluster": cluster,
        })

        tr_cluster = train_df[train_df["demand_cluster"] == cluster]
        if len(tr_cluster) == 0:
            tr_cluster = train_df

        n_rows = len(part)
        print(f"Predict JAN existing pairs | cluster={cluster:>12} rows={n_rows:,} method={rule.get('method')} model={'yes' if model is not None else 'no'}")

        for start in range(0, n_rows, PRED_BATCH_SIZE):
            chunk = part.iloc[start:start + PRED_BATCH_SIZE].copy()
            pf, pi = predict_with_rule(model, tr_cluster, chunk, rule)

            out = chunk[["location", "item_id", "demand_cluster"]].copy()
            out["y_pred_float"] = pf
            out["y_pred_int"] = np.asarray(pi, dtype=int)
            out["quantity"] = np.maximum(out["y_pred_int"].values, 1).astype(np.int32)
            out["pred_source"] = "model_existing_pair"
            out["selected_candidate"] = rule.get("candidate_name", rule.get("method"))
            jan_existing_pred_parts.append(out)

            del chunk, out, pf, pi
            gc.collect()

if jan_existing_pred_parts:
    jan_existing_pred_pd = pd.concat(jan_existing_pred_parts, ignore_index=True)
    jan_existing_pred_pd = jan_existing_pred_pd.drop_duplicates(["location", "item_id"], keep="last")
else:
    jan_existing_pred_pd = pd.DataFrame(columns=["location", "item_id", "demand_cluster", "y_pred_float", "y_pred_int", "quantity", "pred_source", "selected_candidate"])

jan_existing_pred_pd.to_pickle(JAN_EXISTING_PRED_PKL)
print("Jan existing feature-pair predictions:", jan_existing_pred_pd.shape)
print("Saved existing Jan predictions:", JAN_EXISTING_PRED_PKL)

if len(jan_existing_pred_pd):
    display(jan_existing_pred_pd["demand_cluster"].value_counts(dropna=False).rename_axis("demand_cluster").reset_index(name="len"))

jan_existing_pred_pl = (
    pl.from_pandas(jan_existing_pred_pd)
    .with_columns([
        pl.col("location").cast(pl.Utf8, strict=False).alias("location"),
        pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
        pl.col("quantity").cast(pl.Int32, strict=False).fill_null(1).alias("quantity"),
    ])
    .unique(subset=["location", "item_id"], keep="last")
)

del jan_existing_pred_parts, jan_existing_pred_pd
gc.collect()

# -----------------------------
# 2) Build full location x all item keys with Polars
# -----------------------------
START_2025 = pd.Timestamp("2025-01-01")
END_2025_NEXT = pd.Timestamp("2026-01-01")

def month_lit(ts: pd.Timestamp) -> pl.Expr:
    return pl.lit(pd.Timestamp(ts).to_pydatetime()).cast(pl.Datetime)

def parse_datetime_expr(col: str = "updated_date") -> pl.Expr:
    return pl.col(col).cast(pl.Datetime, strict=False)

def scan_locations_2025() -> pl.DataFrame:
    tx_schema = pl.read_parquet_schema(TRANSACTION_PATH)
    for c in ["location", "updated_date"]:
        if c not in tx_schema:
            raise ValueError(f"transaction thiếu cột {c}")

    return (
        pl.scan_parquet(TRANSACTION_PATH)
        .select(["location", "updated_date"])
        .with_columns([
            pl.col("location").cast(pl.Utf8, strict=False).alias("location"),
            parse_datetime_expr("updated_date").alias("updated_date"),
        ])
        .filter(
            (pl.col("updated_date") >= month_lit(START_2025)) &
            (pl.col("updated_date") < month_lit(END_2025_NEXT))
        )
        .select("location")
        .filter(pl.col("location").is_not_null())
        .unique()
        .collect()
        .sort("location")
    )

def scan_tx_items_2025() -> pl.DataFrame:
    tx_schema = pl.read_parquet_schema(TRANSACTION_PATH)
    for c in ["item_id", "updated_date"]:
        if c not in tx_schema:
            raise ValueError(f"transaction thiếu cột {c}")

    return (
        pl.scan_parquet(TRANSACTION_PATH)
        .select(["item_id", "updated_date"])
        .with_columns([
            pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
            parse_datetime_expr("updated_date").alias("updated_date"),
        ])
        .filter(
            (pl.col("updated_date") >= month_lit(START_2025)) &
            (pl.col("updated_date") < month_lit(END_2025_NEXT))
        )
        .select("item_id")
        .filter(pl.col("item_id").is_not_null())
        .unique()
        .collect()
        .sort("item_id")
    )

def scan_items_all() -> pl.DataFrame:
    item_schema = pl.read_parquet_schema(ITEM_PATH)
    if "item_id" not in item_schema:
        raise ValueError("items thiếu cột item_id")

    return (
        pl.scan_parquet(ITEM_PATH)
        .select(pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"))
        .filter(pl.col("item_id").is_not_null())
        .unique()
        .collect()
        .sort("item_id")
    )

locations_pl = scan_locations_2025()
items_table_pl = scan_items_all()
tx_items_pl = scan_tx_items_2025()

all_items_pl = (
    pl.concat([items_table_pl, tx_items_pl], how="vertical")
    .unique(subset=["item_id"], keep="first")
    .sort("item_id")
)

expected_full_rows = locations_pl.height * all_items_pl.height

print("n locations 2025:", f"{locations_pl.height:,}")
print("n items in item table:", f"{items_table_pl.height:,}")
print("n tx items 2025:", f"{tx_items_pl.height:,}")
print("n all items used:", f"{all_items_pl.height:,}")
print("expected full location x all item rows:", f"{expected_full_rows:,}")
print("jan existing predicted pairs:", f"{jan_existing_pred_pl.height:,}")

if locations_pl.height == 0:
    raise ValueError("Không tìm thấy location nào trong transaction 2025.")
if all_items_pl.height == 0:
    raise ValueError("Không tìm thấy item nào trong item table/transaction 2025.")

# -----------------------------
# 3) Export full submission by location chunks
# -----------------------------
if SUBMISSION_CSV.exists():
    SUBMISSION_CSV.unlink()

if DEBUG_PARQUET_DIR.exists():
    shutil.rmtree(DEBUG_PARQUET_DIR)
DEBUG_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

loc_values = locations_pl.get_column("location").to_list()
source_count_parts = []
quantity_count_parts = []
total_rows = 0
first_csv_chunk = True

for chunk_id, start in enumerate(range(0, len(loc_values), CHUNK_LOCATIONS), start=1):
    loc_chunk = pl.DataFrame({"location": loc_values[start:start + CHUNK_LOCATIONS]})

    pair_chunk = (
        loc_chunk
        .join(all_items_pl, how="cross")
        .with_columns([
            pl.col("location").cast(pl.Utf8, strict=False).alias("location"),
            pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
        ])
    )

    out_chunk = (
        pair_chunk
        .join(jan_existing_pred_pl, on=["location", "item_id"], how="left")
        .with_columns([
            pl.when(pl.col("quantity").is_null())
              .then(pl.lit(1))
              .otherwise(pl.col("quantity"))
              .cast(pl.Int32)
              .alias("quantity"),

            pl.when(pl.col("pred_source").is_null())
              .then(pl.lit("missing_full_pair_set_1"))
              .otherwise(pl.col("pred_source"))
              .alias("pred_source"),

            pl.col("demand_cluster").fill_null("cold_start").alias("demand_cluster"),
            pl.col("y_pred_float").fill_null(1.0).alias("y_pred_float"),
            pl.col("y_pred_int").fill_null(1).cast(pl.Int32).alias("y_pred_int"),
        ])
    )

    submit_chunk = (
        out_chunk
        .select([
            pl.col("location").cast(pl.Int32, strict=False).alias("location"),
            pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
            pl.when(pl.col("quantity") < 1)
              .then(pl.lit(1))
              .otherwise(pl.col("quantity"))
              .cast(pl.Int32)
              .alias("quantity"),
        ])
    )

    null_check = submit_chunk.select([
        pl.col("location").is_null().sum().alias("null_location"),
        pl.col("item_id").is_null().sum().alias("null_item_id"),
        pl.col("quantity").is_null().sum().alias("null_quantity"),
    ]).row(0)
    if null_check != (0, 0, 0):
        raise ValueError(f"Chunk {chunk_id} có null sau cast: {null_check}")

    # CSV append bằng file object để không giữ full DataFrame trong RAM.
    mode = "wb" if first_csv_chunk else "ab"
    with open(SUBMISSION_CSV, mode) as f:
        submit_chunk.write_csv(f, include_header=first_csv_chunk)
    first_csv_chunk = False

    debug_chunk = out_chunk.select([
        pl.col("location").cast(pl.Int32, strict=False).alias("location"),
        pl.col("item_id").cast(pl.Utf8, strict=False).alias("item_id"),
        pl.when(pl.col("quantity") < 1)
              .then(pl.lit(1))
              .otherwise(pl.col("quantity"))
              .cast(pl.Int32)
              .alias("quantity"),
        "pred_source",
        "demand_cluster",
        "y_pred_float",
        "y_pred_int",
    ])
    debug_chunk.write_parquet(DEBUG_PARQUET_DIR / f"part_{chunk_id:04d}.parquet", compression="zstd")

    source_count_parts.append(debug_chunk.group_by("pred_source").len())
    quantity_count_parts.append(submit_chunk.group_by("quantity").len())

    total_rows += submit_chunk.height
    print(f"chunk {chunk_id:04d}: locations={loc_chunk.height:,} rows={submit_chunk.height:,} total={total_rows:,}")

    del loc_chunk, pair_chunk, out_chunk, submit_chunk, debug_chunk
    gc.collect()

if total_rows != expected_full_rows:
    raise ValueError(f"Sai số dòng final: got={total_rows:,}, expected={expected_full_rows:,}")

source_counts = (
    pl.concat(source_count_parts, how="vertical")
    .group_by("pred_source")
    .agg(pl.col("len").sum().alias("len"))
    .sort("len", descending=True)
)

quantity_counts = (
    pl.concat(quantity_count_parts, how="vertical")
    .group_by("quantity")
    .agg(pl.col("len").sum().alias("len"))
    .sort("quantity")
)

print("\nForecast source distribution:")
display(source_counts)

print("\nQuantity distribution head:")
display(quantity_counts.head(50))

print("\n===== SAVED OUTPUTS =====")
print("DEC valid predictions:", DEC_PRED_PATH)
print("Model bundle:", MODEL_BUNDLE_PATH)
print("JAN submission CSV:", SUBMISSION_CSV)
print("JAN debug parquet parts:", DEBUG_PARQUET_DIR)
print("Rows:", f"{total_rows:,}")

# Optional full PKL. This can crash on ~30M rows, so default False.
if CREATE_FULL_PKL:
    print("\nCREATE_FULL_PKL=True: reading full CSV into pandas. This may require a lot of RAM.")
    submission_pd = pd.read_csv(
        SUBMISSION_CSV,
        dtype={"location": "int32", "item_id": "object", "quantity": "int32"},
    )
    submission_pd = submission_pd[["location", "item_id", "quantity"]]
    submission_pd.columns = pd.Index(["location", "item_id", "quantity"], dtype=object)
    submission_pd.index = pd.RangeIndex(start=0, stop=len(submission_pd), step=1)
    submission_pd.to_pickle(SUBMISSION_PKL)
    print("JAN submission PKL:", SUBMISSION_PKL)
    print("PKL shape:", submission_pd.shape)
    display(submission_pd.head(10))
    del submission_pd
else:
    print("\nCREATE_FULL_PKL=False nên chưa tạo PKL full để tránh crash.")
    print("Nếu máy đủ RAM và bắt buộc cần PKL, chạy lại cell này sau khi set: CREATE_FULL_PKL = True")

gc.collect()


Predict JAN existing pairs | cluster=  cold_start rows=1,688,434 method=constant_one model=no
Predict JAN existing pairs | cluster=intermittent rows=599,482 method=candidate_integer model=yes
Predict JAN existing pairs | cluster=  low_volume rows=324,493 method=candidate_integer model=yes
Predict JAN existing pairs | cluster=     regular rows=532,149 method=candidate_integer model=yes
Jan existing feature-pair predictions: (3144558, 8)
Saved existing Jan predictions: /kaggle/working/cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601/jan_2026_existing_feature_pair_predictions.pkl


,demand_cluster,len
0,cold_start,1688434
1,intermittent,599482
2,regular,532149
3,low_volume,324493


n locations 2025: 1,038
n items in item table: 29,823
n tx items 2025: 20,393
n all items used: 29,862
expected full location x all item rows: 30,996,756
jan existing predicted pairs: 3,144,558
chunk 0001: locations=50 rows=1,493,100 total=1,493,100
chunk 0002: locations=50 rows=1,493,100 total=2,986,200
chunk 0003: locations=50 rows=1,493,100 total=4,479,300
chunk 0004: locations=50 rows=1,493,100 total=5,972,400
chunk 0005: locations=50 rows=1,493,100 total=7,465,500
chunk 0006: locations=50 rows=1,493,100 total=8,958,600
chunk 0007: locations=50 rows=1,493,100 total=10,451,700
chunk 0008: locations=50 rows=1,493,100 total=11,944,800
chunk 0009: locations=50 rows=1,493,100 total=13,437,900
chunk 0010: locations=50 rows=1,493,100 total=14,931,000
chunk 0011: locations=50 rows=1,493,100 total=16,424,100
chunk 0012: locations=50 rows=1,493,100 total=17,917,200
chunk 0013: locations=50 rows=1,493,100 total=19,410,300
chunk 0014: locations=50 rows=1,493,100 total=20,903,400
chunk 0015: lo

pred_source,len
str,u32
"""missing_full_pair_set_1""",27852198
"""model_existing_pair""",3144558



Quantity distribution head:


quantity,len
i32,u32
1,30530500
2,219949
3,69223
4,42461
5,29184
6,19986
7,13577
8,10216
9,7685



===== SAVED OUTPUTS =====
DEC valid predictions: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/saved_models/dec_2025_valid_predictions_train_202501_202511.pkl
Model bundle: /kaggle/working/cs116_cache_SIMPLE_4CLUSTER_WIDE_INTERMITTENT_train_202501_202511_valid_202512/saved_models/simple_4cluster_wide_intermit_lgbm_xgb_train_202501_202511_valid_202512.joblib
JAN submission CSV: /kaggle/working/cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601/submission_sale_forecasting_train_202501_202511_valid_202512_predict_202601.csv
JAN debug parquet parts: /kaggle/working/cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601/jan_2026_debug_parquet_parts
Rows: 30,996,756

CREATE_FULL_PKL=True: reading full CSV into pandas. This may require a lot of RAM.
JAN submission PKL: /kaggle/working/cs116_outputs_4cluster_wide_intermit_train_202501_202511_valid_202512_predict_202601/submission_s

,location,item_id,quantity
0,1000,0000280000019,1
1,1000,0000280000020,1
2,1000,0000280000048,1
3,1000,0000280000050,1
4,1000,0000280000070,1
5,1000,0000280000089,1
6,1000,0000280000092,1
7,1000,0000280000098,1
8,1000,0000280000099,1
9,1000,0000280000100,1


0